# Sarcasm-R1 untuk Deteksi Sarkasme Bahasa Indonesia (IdSarcasm: Twitter & Reddit)

Notebook ini mengimplementasikan secara penuh metodologi **Sarcasm-R1: Enhancing Sarcasm Detection through Focused Reasoning** ([Yang et al., EMNLP 2025](https://github.com/yangqi1725/Sarcasm-R1)) pada kedua benchmark **IdSarcasm** ([Suhartono et al., IEEE Access 2024](https://doi.org/10.1109/ACCESS.2024.3416955)):
1. **`w11wo/twitter_indonesia_sarcastic`** (`1.878 train`, `268 validation`, `538 test`)
2. **`w11wo/reddit_indonesia_sarcastic`** (`9.881 train`, `1.411 validation`, `2.824 test`)

### Cakupan Eksperimen Tesis S2 di Notebook Ini:
- **Bagian 3.3 & Tabel 1 (`IdSarcasm-Reason`)**: Sintesis penalaran 3 dimensi (`(1)language`, `(2)context`, `(3)emotion`), pembuatan sampel negatif bernoise ($y^-$), pasangan alasan juri $\mathcal{D}_{\text{CoT}}$ (**Persamaan 5**), analisis kata budaya lokal Indonesia (**Tabel 8 IdSarcasm**), dan statistik panjang penalaran (**Tabel 1 Sarcasm-R1**).
- **Bagian 3.2 & Figure 4 (`SarGRM`: `LinGRM`, `ConGRM`, `EmoGRM`)**: Evaluasi skalabilitas *majority voting* $K \in \{1, 2, 4, 8, 16, 32\}$ (**Persamaan 6 & RQ3**).
- **Bagian 3.1 & Appendix A (3-Stage GRPO `Sarcasm-R1` — RQ1)**:
  - **Stage 1**: *Format Reward* (`+1.0` / `0.0`)
  - **Stage 2**: *Format Reward* (`0.0` / `-1.0`) + 3 *Reward Models* (`LinGRM`, `ConGRM`, `EmoGRM`: `+0.5` / `0.0`)
  - **Stage 3**: *Format Reward* (`0.0` / `-1.0`) + 3 *Reward Models* (`0.0` / `-0.5`) + *Correctness Reward* (`+1.5` / `0.0`)
- **Bagian 4.2 & Tabel 3–4 (Studi Ablasi 3 Dimensi — RQ2)**: Eksperimen `w/o Lin`, `w/o Con`, dan `w/o Emo` beserta uji signifikansi statistik ($t$-statistic & $p$-value).

> **Fitur Hemat Compute Unit (CU) Google Colab Pro (A100)**:
> - **Auto-Resume & Cache di Google Drive**: Semua data dan hasil evaluasi disimpan di `/content/drive/MyDrive/idsarcasm/sarcasm_r1/`. Tahap yang sudah selesai otomatis di-*skip* jika notebook dijalankan ulang.
> - **Auto-Disconnect Runtime**: Di akhir eksekusi, notebook otomatis memanggil `runtime.unassign()` agar GPU A100 langsung dimatikan dan tidak menyedot Compute Unit saat ditinggal.

## 1. Periksa Runtime & Konfigurasi Utama Eksperimen

Anda dapat menjalankan Bagian 1–4 pada **CPU Runtime (0 CU)** terlebih dahulu untuk menyiapkan dataset `IdSarcasm-Reason` di Google Drive, lalu mengganti ke **GPU A100** untuk Bagian 5–9, atau langsung menjalankan seluruh notebook pada **GPU A100**.

In [ ]:
import os
import torch

HAS_GPU = torch.cuda.is_available()
print("PyTorch:", torch.__version__)
if HAS_GPU:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Aktif: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GiB | bf16: {torch.cuda.is_bf16_supported()}")
else:
    print("Mode Runtime: CPU (0 Compute Unit - Cocok untuk Persiapan Data Bagian 1-4 & Pembuatan Laporan Bagian 9)")

## 2. Instal Dependensi Colab

In [ ]:
%pip -q install "transformers>=4.47.1" "trl>=0.14.0" "peft>=0.14.0" datasets evaluate accelerate bitsandbytes scikit-learn scipy matplotlib pandas google-genai

## 3. Mount Google Drive & Pengaturan Parameter Eksperimen

Ubah parameter di bawah ini sesuai tahapan yang ingin Anda jalankan:
- `DATASETS_TO_RUN = ["twitter", "reddit"]` (jalankan Twitter dulu yang cepat, lalu otomatis lanjut ke Reddit)
- `BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"` (sesuai Appendix A paper Sarcasm-R1; bisa diganti `"Qwen/Qwen2.5-3B-Instruct"` jika menggunakan GPU lebih kecil seperti T4/L4)
- `AUTO_DISCONNECT_A100 = True` (otomatis mematikan runtime Colab di akhir notebook untuk menghemat Compute Unit)

In [ ]:
from pathlib import Path
import json, gc, shutil, time

try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    ROOT_OUT = Path("/content/drive/MyDrive/idsarcasm/sarcasm_r1")
    try:
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY") or ""
    except Exception:
        pass
except Exception:
    ROOT_OUT = Path("./outputs_sarcasm_r1")

DATA_DIR = ROOT_OUT / "data"
RUN_DIR = ROOT_OUT / "runs"
ASSET_DIR = ROOT_OUT / "assets"
WORK_DIR = Path("/content/sarcasm_r1_workspace")

for p in (DATA_DIR, RUN_DIR, ASSET_DIR, WORK_DIR):
    p.mkdir(parents=True, exist_ok=True)

# Konfigurasi Eksperimen Tesis S2
DATASETS_TO_RUN = ["twitter", "reddit"]  # Evaluasi terpisah sesuai Tabel 1 & Tabel 2 IdSarcasm
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"   # Model kebijakan sesuai Appendix A Paper Sarcasm-R1
REDDIT_GRPO_SUBSET = 2000                 # Stratified subset untuk menghemat Compute Unit pada rollout GRPO Reddit
SEED = 42

# Hyperparameter 3-Stage GRPO (Appendix A Hal. 12 & config.yaml)
STAGE1_STEPS = 30   # Stage 1: Format reward only (+1 / 0)
STAGE2_STEPS = 60   # Stage 2: Format (0 / -1) + LinGRM, ConGRM, EmoGRM (+0.5 / 0)
STAGE3_STEPS = 120  # Stage 3: Format (0 / -1) + LinGRM, ConGRM, EmoGRM (0 / -0.5) + Correctness (+1.5 / 0)
ABLATION_STEPS = 60 # Langkah Stage 2-3 untuk masing-masing run ablasi (w/o Lin, w/o Con, w/o Emo)

RUN_ZERO_AND_FEW_SHOT_BASELINES = True
RUN_ABLATION_STUDY_ON_TWITTER = True
AUTO_DISCONNECT_A100 = True

print("Direktori Output Google Drive:", ROOT_OUT)
print("Direktori Kerja Colab:", WORK_DIR)

## 4. Tulis Modul Mandiri (`prepare_idsarcasm_reason.py`, `train_sargrm.py`, `train_idsarcasm_grpo.py`, `config_colab_a100.yaml`)

Sel ini menulis seluruh modul kode **Sarcasm-R1** langsung ke `/content/sarcasm_r1_workspace/` sehingga notebook ini 100% mandiri (*self-contained*) di Google Colab.

In [ ]:
from pathlib import Path
import sys

GRPO_DIR = WORK_DIR / "code" / "grpo"
RM_DIR = WORK_DIR / "code" / "reward_model"
GRPO_DIR.mkdir(parents=True, exist_ok=True)
RM_DIR.mkdir(parents=True, exist_ok=True)

(GRPO_DIR / "prepare_idsarcasm_reason.py").write_text('#!/usr/bin/env python\n# -*- coding: utf-8 -*-\n"""\nPersiapan Dataset IdSarcasm-Reason (Adaptasi Bagian 3.3 Paper Sarcasm-R1 EMNLP 2025\nuntuk Benchmark IdSarcasm IEEE Access 2024: Twitter & Reddit Indonesia).\n\nMencakup 3 langkah utama sesuai Bagian 3.3 Paper Sarcasm-R1:\n1. Standard Reasoning Generation (y+): Penalaran 3 dimensi (1)language, (2)context, (3)emotion (Figure 3).\n2. Negative Sample Generation (y-): Penalaran bernoise (menghapus info krusial dimensi atau mencampur antar dimensi).\n3. Data Validation & Judgment CoT (D_CoT, Persamaan 5): Pasangan (x, y+, y-, I_CoT) -> (v_CoT, I, \'First\'/\'Second\').\n\nMenghasilkan:\n- CSV kompatibel dengan `sarcasmr1/Sarcasm-R1/code/grpo/train.py` (kolom: question, answer, label, text, dll.)\n- Data latihan/evaluasi SarGRM (`d_cot_lin.jsonl`, `d_cot_con.jsonl`, `d_cot_emo.jsonl`)\n- Statistik Tabel 1 Paper Sarcasm-R1 (`#Train`, `#Test`, `#Avg.Len`, `#Lin.Avg`, `#Con.Avg`, `#Emo.Avg`, `%Sarcasm`)\n- Statistik kata lokal budaya Indonesia (Tabel 8 Paper IdSarcasm)\n"""\n\nimport argparse\nimport hashlib\nimport json\nimport os\nimport random\nimport re\nimport time\nfrom collections import Counter\nfrom pathlib import Path\nfrom typing import Any\n\nimport pandas as pd\n\n# Daftar kata lokal budaya Indonesia dari Tabel 8 Paper IdSarcasm (Suhartono et al., IEEE Access 2024)\nIDSARCASM_TABLE8_LOCAL_WORDS = {\n    "food": ["indomie", "rendang", "tempe", "gule", "sate"],\n    "transportation": ["angkot", "ojol", "gojek"],\n    "religion": ["doa", "gaib", "alhamdulillah", "insyaallah"],\n    "adjective": ["bule", "santun", "alay"],\n}\nALL_LOCAL_WORDS = {w for words in IDSARCASM_TABLE8_LOCAL_WORDS.values() for w in words}\n\n# Prompt Asli Figure 3 Paper Sarcasm-R1 (diadaptasi untuk teks Indonesia namun menjaga tag & struktur)\nFIGURE3_SYNTHESIS_PROMPT = """Here is an Indonesian social media text that is {not_str}a sarcastic text.\nPlease deduce the reason why it is {not_str}sarcastic from the perspectives of (1)language, (2)context, and (3)emotion in this text. Please answer in the following format:\n<reason>\n(1)language: like keywords, rhetorical devices, punctuation and language style\n(2)context: like topic, Indonesian cultural background, social consensus, common knowledge\n(3)emotion: like emotional words, special symbols and emotional reversal\n</reason>\nHere is the text: {text}"""\n\n# Prompt Inferensi & Pelatihan GRPO (Menyatukan Figure 3 Paper & Parser `{thinking_content} {response}` di train.py)\nGRPO_QUESTION_TEMPLATE = """Here is an Indonesian social media text from {platform}. Please deduce the reason why it is or is not sarcastic from the perspectives of (1)language, (2)context, and (3)emotion, and then give the final prediction ("sarcastic" or "not sarcastic").\nPlease answer in the following format with two curly-brace blocks:\n{{\n<reason>\n(1)language: analyze keywords, rhetorical devices, punctuation, and Indonesian language style\n(2)context: analyze topic, Indonesian cultural background, social consensus, and common knowledge\n(3)emotion: analyze emotional words, special symbols, and emotional reversal\n</reason>\n}}\n{{sarcastic or not sarcastic}}\n\nHere is the text: {text}"""\n\n# Konstanta Instruksi CoT SarGRM sesuai Persamaan (5) Hal. 5 Paper Sarcasm-R1\nI_COT_PROMPT = "Let\'s think about which of these two sarcastic reasons is better step by step."\nI_JUDGE_PROMPT = "Between the two sarcastic reasons above, which is better(First / Second)?"\n\nPOSITIVE_PRAISE_WORDS = {\n    "terima kasih", "makasih", "hebat", "bagus", "mantap", "pintar", "pinter", "jenius",\n    "luar biasa", "indah", "senang", "bahagia", "syukur", "alhamdulillah", "sukses",\n    "keren", "adil", "makmur", "sejahtera", "cepat", "ramah", "santun", "berkah", "enak",\n}\nNEGATIVE_SITUATION_WORDS = {\n    "banjir", "macet", "korupsi", "mahal", "rusak", "bohong", "maling", "telat", "lambat",\n    "hancur", "susah", "miskin", "hutang", "utang", "kabur", "razia", "celaka", "bodoh",\n    "parah", "ancur", "ngaco", "gagal", "mati", "sakit", "bencana", "sampah", "polusi",\n}\nSLANG_AND_MARKERS = {\n    "banget", "bgt", "dong", "deh", "sih", "kok", "nih", "tuh", "wkwk", "haha", "hehe",\n    ":v", "anjir", "astaga", "yaelah", "cie", "mantul", "<hashtag>", "<username>", "<link>",\n}\n\n\ndef extract_3d_sections(reason_text: str) -> dict[str, str]:\n    """Ekstrak komponen (1)language, (2)context, dan (3)emotion dari blok <reason>."""\n    cleaned = re.sub(r"</?reason>", "", reason_text, flags=re.IGNORECASE).strip()\n    lin_match = re.search(\n        r"\\(1\\)\\s*language\\s*:\\s*(.*?)(?=\\(2\\)\\s*context\\s*:|$)",\n        cleaned,\n        flags=re.IGNORECASE | re.DOTALL,\n    )\n    con_match = re.search(\n        r"\\(2\\)\\s*context\\s*:\\s*(.*?)(?=\\(3\\)\\s*emotion\\s*:|$)",\n        cleaned,\n        flags=re.IGNORECASE | re.DOTALL,\n    )\n    emo_match = re.search(\n        r"\\(3\\)\\s*emotion\\s*:\\s*(.*?)$",\n        cleaned,\n        flags=re.IGNORECASE | re.DOTALL,\n    )\n    return {\n        "lin": lin_match.group(1).strip() if lin_match else "",\n        "con": con_match.group(1).strip() if con_match else "",\n        "emo": emo_match.group(1).strip() if emo_match else "",\n    }\n\n\ndef analyze_indonesian_cues(text: str, label: int, platform: str) -> dict[str, str]:\n    """\n    Membangun penalaran 3 dimensi standar (y+) berbasis fitur linguistik, konteks budaya Indonesia\n    (termasuk kata lokal Tabel 8 IdSarcasm), dan pembalikan emosi (emotional reversal).\n    Digunakan sebagai fallback terstruktur atau pendamping sintesis API LLM.\n    """\n    tokens = re.findall(r"[a-zA-Z0-9_<>]+", text.lower())\n    token_set = set(tokens)\n\n    found_pos = sorted(w for w in POSITIVE_PRAISE_WORDS if w in text.lower())\n    found_neg = sorted(w for w in NEGATIVE_SITUATION_WORDS if w in text.lower())\n    found_slang = sorted(w for w in SLANG_AND_MARKERS if w in text.lower())\n    found_local = sorted(w for w in ALL_LOCAL_WORDS if w in token_set)\n\n    has_multi_punct = bool(re.search(r"[?!.]{2,}", text))\n    has_quotes = bool(re.search(r"[\\"\']", text))\n    has_emoticon = bool(re.search(r"(:v|wkwk|haha|\\?\\?+|!!+)", text.lower()))\n\n    # 1. Linguistic Dimension\n    lin_cues = []\n    if found_pos:\n        lin_cues.append(f"hyperbolic/praise expressions ({\', \'.join(found_pos[:3])})")\n    if found_slang:\n        lin_cues.append(f"colloquial Indonesian markers ({\', \'.join(found_slang[:3])})")\n    if has_multi_punct or has_emoticon:\n        lin_cues.append("emphatic punctuation/emoticons signaling ironic intonation")\n    if has_quotes:\n        lin_cues.append("quotation marks highlighting ironic or contradictory phrasing")\n    if not lin_cues:\n        lin_cues.append("direct Indonesian conversational phrasing without overt ironic exaggeration")\n\n    if label == 1:\n        lin_text = (\n            f"The text employs {\', \'.join(lin_cues)} as rhetorical devices to frame a sharp "\n            f"ironic statement typical of Indonesian {platform} discourse."\n        )\n    else:\n        lin_text = (\n            f"The text uses {\', \'.join(lin_cues)} in a literal manner where lexical choices "\n            f"directly match the speaker\'s intended meaning."\n        )\n\n    # 2. Contextual Dimension\n    local_note = (\n        f" featuring local Indonesian cultural terms ({\', \'.join(found_local)})"\n        if found_local\n        else ""\n    )\n    if label == 1:\n        if found_neg:\n            con_text = (\n                f"In the Indonesian social media context ({platform}){local_note}, the situation "\n                f"involves negative realities ({\', \'.join(found_neg[:3])}) that violate public "\n                f"expectations and common sense, exposing social criticism."\n            )\n        else:\n            con_text = (\n                f"Within the Indonesian socio-cultural context on {platform}{local_note}, common sense "\n                f"and shared public consensus indicate an unfavorable or absurd situation behind the statement."\n            )\n    else:\n        con_text = (\n            f"Within the Indonesian {platform} context{local_note}, the topic reflects a standard "\n            f"everyday discussion or opinion that aligns with common sense without situational incongruity."\n        )\n\n    # 3. Emotional Dimension\n    if label == 1:\n        if found_pos and found_neg:\n            emo_text = (\n                f"Because there is a strong emotional reversal between the surface praise "\n                f"({\', \'.join(found_pos[:2])}) and the negative event ({\', \'.join(found_neg[:2])}), "\n                f"positive phrasing masks negative dissatisfaction and sarcastic intent."\n            )\n        else:\n            emo_text = (\n                "Because the surface expression contrasts with the underlying frustration or mockery, "\n                "this emotional reversal uncovers the hidden negative sarcastic intent."\n            )\n    else:\n        emo_text = (\n            "Because the emotional expression is consistent with the literal message without any "\n            "emotional reversal or hidden dissatisfaction, the text is not sarcastic."\n        )\n\n    return {"lin": lin_text, "con": con_text, "emo": emo_text}\n\n\ndef generate_noisy_negative_reasoning(\n    pos_3d: dict[str, str], label: int, rng: random.Random\n) -> tuple[dict[str, str], dict[str, str]]:\n    """\n    Bagian 3.3.2 Paper Sarcasm-R1: Negative Sample Generation (y-).\n    Menggunakan 2 prinsip dari paper:\n    (1) Penalaran tidak secara akurat mengekspresikan isi dimensi tersebut (menghapus info krusial / membalik klaim).\n    (2) Penalaran tercampur dengan isi dari dimensi lain (cross-dimension confusion).\n    Mengembalikan (neg_3d, modification_reasons_J).\n    """\n    strategy = rng.choice(["confused_dimension", "corrupted_content"])\n    neg_3d = {}\n    reasons_j = {}\n\n    if strategy == "confused_dimension":\n        # Prinsip (2): Mencampur isi antar dimensi\n        neg_3d["lin"] = pos_3d["con"]\n        neg_3d["con"] = pos_3d["emo"]\n        neg_3d["emo"] = (\n            "The text simply uses words and sentences without analyzing any emotional contrast "\n            "or sentiment polarity reversal."\n        )\n        reasons_j["lin"] = (\n            "The positive reasoning accurately analyzes linguistic keywords, punctuation, and rhetorical "\n            "style, whereas the negative reasoning confuses the linguistic dimension with background context."\n        )\n        reasons_j["con"] = (\n            "The positive reasoning properly evaluates the Indonesian socio-cultural context and common sense, "\n            "whereas the negative reasoning confuses context with emotional descriptions."\n        )\n        reasons_j["emo"] = (\n            "The positive reasoning clearly identifies the emotional reversal (or consistency) central to "\n            "sarcasm detection, while the negative reasoning omits critical emotional contrast analysis."\n        )\n    else:\n        # Prinsip (1): Menghapus informasi krusial & menambahkan klaim yang keliru\n        neg_3d["lin"] = "The text contains standard words with no notable rhetorical or punctuation cues."\n        neg_3d["con"] = "There is no specific social topic or cultural background relevant to this sentence."\n        if label == 1:\n            neg_3d["emo"] = (\n                "The speaker expresses genuine literal happiness with no emotional reversal or hidden criticism."\n            )\n        else:\n            neg_3d["emo"] = (\n                "The speaker creates a sharp emotional reversal by praising a catastrophic failure."\n            )\n        reasons_j["lin"] = (\n            "The positive reasoning captures specific lexical and rhetorical cues in the Indonesian text, "\n            "while the negative sample removes critical linguistic information."\n        )\n        reasons_j["con"] = (\n            "The positive reasoning integrates topic and Indonesian cultural commonsense, whereas the negative "\n            "sample fails to provide relevant contextual grounding."\n        )\n        reasons_j["emo"] = (\n            "The positive reasoning accurately reflects the true emotional reversal mechanism of the text, "\n            "whereas the negative sample hallucinates an incorrect emotional polarity."\n        )\n\n    return neg_3d, reasons_j\n\n\ndef build_d_cot_entries(\n    text: str,\n    pos_3d: dict[str, str],\n    neg_3d: dict[str, str],\n    reasons_j: dict[str, str],\n) -> dict[str, list[dict[str, str]]]:\n    """\n    Membangun data latihan/evaluasi SarGRM (D_CoT) sesuai Persamaan (5) pada Hal. 5 Paper Sarcasm-R1:\n    D_CoT = {((x, y+, y-, I_CoT), (v_CoT, I, \'First\'))} U {((x, y-, y+, I_CoT), (v_CoT, I, \'Second\'))}\n    untuk masing-masing dimensi: LinGRM, ConGRM, dan EmoGRM.\n    """\n    d_cot_by_dim: dict[str, list[dict[str, str]]] = {"lin": [], "con": [], "emo": []}\n    dim_names = {"lin": "linguistic", "con": "contextual", "emo": "emotional"}\n\n    for dim_key, dim_label in dim_names.items():\n        y_pos = pos_3d[dim_key]\n        y_neg = neg_3d[dim_key]\n        j_exp = reasons_j[dim_key]\n\n        # Pasangan 1: (x, y+, y-, I_CoT) -> First menang\n        input_first = (\n            f"Original text: {text}\\n"\n            f"First {dim_label} reasoning: {y_pos}\\n"\n            f"Second {dim_label} reasoning: {y_neg}\\n"\n            f"{I_COT_PROMPT}"\n        )\n        v_cot_first = (\n            f"Step 1: Evaluate the First {dim_label} reasoning. {j_exp} "\n            f"Step 2: Compare both candidates. Therefore, the First reasoning is superior."\n        )\n        target_first = f"{v_cot_first}\\n{I_JUDGE_PROMPT}\\nFirst"\n\n        # Pasangan 2: (x, y-, y+, I_CoT) -> Second menang\n        input_second = (\n            f"Original text: {text}\\n"\n            f"First {dim_label} reasoning: {y_neg}\\n"\n            f"Second {dim_label} reasoning: {y_pos}\\n"\n            f"{I_COT_PROMPT}"\n        )\n        v_cot_second = (\n            f"Step 1: Evaluate both {dim_label} reasonings. {j_exp} "\n            f"Step 2: Compare both candidates. Therefore, the Second reasoning is superior."\n        )\n        target_second = f"{v_cot_second}\\n{I_JUDGE_PROMPT}\\nSecond"\n\n        d_cot_by_dim[dim_key].append(\n            {\n                "dimension": dim_key,\n                "text": text,\n                "y_first": y_pos,\n                "y_second": y_neg,\n                "prompt": input_first,\n                "completion": target_first,\n                "v_cot": v_cot_first,\n                "gold_choice": "First",\n            }\n        )\n        d_cot_by_dim[dim_key].append(\n            {\n                "dimension": dim_key,\n                "text": text,\n                "y_first": y_neg,\n                "y_second": y_pos,\n                "prompt": input_second,\n                "completion": target_second,\n                "v_cot": v_cot_second,\n                "gold_choice": "Second",\n            }\n        )\n\n    return d_cot_by_dim\n\n\ndef synthesize_with_gemini_if_available(\n    rows: list[dict[str, Any]],\n    platform: str,\n    api_key: str | None,\n    cache_path: Path,\n    max_api_samples: int = 500,\n) -> dict[str, dict[str, str]]:\n    """\n    Opsional: Menggunakan Gemini API (Google AI Pro) di CPU untuk men-generate penalaran 3 dimensi (Figure 3).\n    Menyimpan progres secara bertahap ke `cache_path` agar bisa di-resume kapan saja.\n    """\n    cached: dict[str, dict[str, str]] = {}\n    if cache_path.exists():\n        try:\n            cached = json.loads(cache_path.read_text(encoding="utf-8"))\n        except Exception:\n            cached = {}\n\n    if not api_key:\n        return cached\n\n    try:\n        from google import genai\n        client = genai.Client(api_key=api_key)\n    except Exception as exc:\n        print(f"[Info] google-genai SDK tidak tersedia ({exc}); menggunakan ekstraktor 3D terstruktur.")\n        return cached\n\n    new_count = 0\n    for idx, row in enumerate(rows[:max_api_samples]):\n        row_hash = hashlib.sha256(f"{platform}:{row[\'text\']}".encode("utf-8")).hexdigest()[:16]\n        if row_hash in cached:\n            continue\n\n        not_str = "" if int(row["label"]) == 1 else "not "\n        prompt = FIGURE3_SYNTHESIS_PROMPT.format(not_str=not_str, text=row["text"])\n        try:\n            resp = client.models.generate_content(\n                model="gemini-2.0-flash",\n                contents=prompt,\n            )\n            parsed = extract_3d_sections(resp.text or "")\n            if parsed["lin"] and parsed["con"] and parsed["emo"]:\n                cached[row_hash] = parsed\n                new_count += 1\n                if new_count % 25 == 0:\n                    cache_path.write_text(json.dumps(cached, ensure_ascii=False, indent=2), encoding="utf-8")\n                time.sleep(0.15)\n        except Exception as err:\n            print(f"[Warning] Gemini API berhenti pada indeks {idx} ({err}); melanjutkan dengan cache/fallback.")\n            break\n\n    if new_count > 0:\n        cache_path.write_text(json.dumps(cached, ensure_ascii=False, indent=2), encoding="utf-8")\n    return cached\n\n\ndef word_len(text: str) -> int:\n    return len(text.split())\n\n\ndef prepare_dataset_splits(\n    dataset_key: str,\n    output_dir: Path,\n    seed: int = 42,\n    gemini_api_key: str | None = None,\n    max_train_grpo_samples: int | None = None,\n    max_d_cot_pairs_per_split: int = 600,\n) -> dict[str, Any]:\n    """\n    Menyiapkan dataset `twitter` atau `reddit` ke format CSV Sarcasm-R1 + D_CoT SarGRM + Tabel 1 Statistik.\n    """\n    if dataset_key == "twitter":\n        hf_name = "w11wo/twitter_indonesia_sarcastic"\n        text_col = "tweet"\n        platform_label = "Twitter"\n    elif dataset_key == "reddit":\n        hf_name = "w11wo/reddit_indonesia_sarcastic"\n        text_col = "text"\n        platform_label = "Reddit"\n    else:\n        raise ValueError(f"dataset_key tidak dikenal: {dataset_key}")\n\n    ds_out_dir = output_dir / dataset_key\n    ds_out_dir.mkdir(parents=True, exist_ok=True)\n\n    from datasets import load_dataset\n    raw = load_dataset(hf_name)\n    rng = random.Random(seed)\n\n    # Opsional sintesis Gemini API (berjalan di CPU, disimpan ke cache)\n    train_raw_rows = [{"text": str(r[text_col]), "label": int(r["label"])} for r in raw["train"]]\n    gemini_cache = synthesize_with_gemini_if_available(\n        train_raw_rows,\n        platform=platform_label,\n        api_key=gemini_api_key,\n        cache_path=ds_out_dir / "gemini_3d_cache.json",\n    )\n\n    split_dfs: dict[str, pd.DataFrame] = {}\n    d_cot_splits: dict[str, dict[str, list[dict[str, str]]]] = {}\n    local_word_counter = Counter()\n\n    for split_name in ("train", "validation", "test"):\n        split_data = raw[split_name]\n        records = []\n        d_cot_accum: dict[str, list[dict[str, str]]] = {"lin": [], "con": [], "emo": []}\n\n        for idx, item in enumerate(split_data):\n            text = str(item[text_col]).strip()\n            label = int(item["label"])\n            answer_str = "sarcastic" if label == 1 else "not sarcastic"\n\n            # Hitung kata lokal budaya Indonesia (Tabel 8 IdSarcasm)\n            tokens_lower = set(re.findall(r"[a-zA-Z0-9_]+", text.lower()))\n            matched_local = sorted(w for w in ALL_LOCAL_WORDS if w in tokens_lower)\n            for w in matched_local:\n                local_word_counter[w] += 1\n\n            row_hash = hashlib.sha256(f"{platform_label}:{text}".encode("utf-8")).hexdigest()[:16]\n            if row_hash in gemini_cache:\n                pos_3d = gemini_cache[row_hash]\n            else:\n                pos_3d = analyze_indonesian_cues(text, label, platform_label)\n\n            neg_3d, reasons_j = generate_noisy_negative_reasoning(pos_3d, label, rng)\n\n            formatted_pos_reason = (\n                "<reason>\\n"\n                f"(1)language: {pos_3d[\'lin\']}\\n"\n                f"(2)context: {pos_3d[\'con\']}\\n"\n                f"(3)emotion: {pos_3d[\'emo\']}\\n"\n                "</reason>"\n            )\n            sft_target = f"{{\\n{formatted_pos_reason}\\n}}\\n{{{answer_str}}}"\n\n            records.append(\n                {\n                    "id": f"{dataset_key}_{split_name}_{idx}",\n                    "platform": platform_label,\n                    "text": text,\n                    "label": label,\n                    "question": GRPO_QUESTION_TEMPLATE.format(platform=platform_label, text=text),\n                    "answer": answer_str,\n                    "sft_completion": sft_target,\n                    "lin_pos": pos_3d["lin"],\n                    "con_pos": pos_3d["con"],\n                    "emo_pos": pos_3d["emo"],\n                    "lin_neg": neg_3d["lin"],\n                    "con_neg": neg_3d["con"],\n                    "emo_neg": neg_3d["emo"],\n                    "has_local_word": len(matched_local) > 0,\n                    "local_words": ",".join(matched_local),\n                }\n            )\n\n            if len(d_cot_accum["lin"]) < max_d_cot_pairs_per_split:\n                pair_entries = build_d_cot_entries(text, pos_3d, neg_3d, reasons_j)\n                for dim_key in ("lin", "con", "emo"):\n                    d_cot_accum[dim_key].extend(pair_entries[dim_key])\n\n        df = pd.DataFrame(records)\n        csv_path = ds_out_dir / f"{split_name}.csv"\n        df.to_csv(csv_path, index=False, encoding="utf-8")\n        split_dfs[split_name] = df\n        d_cot_splits[split_name] = d_cot_accum\n\n    # Jika max_train_grpo_samples ditentukan (misal untuk menghemat CU pada Reddit),\n    # buat stratified subset khusus rollout GRPO dengan tetap mempertahankan rasio asli 1:3\n    train_df = split_dfs["train"]\n    if max_train_grpo_samples and len(train_df) > max_train_grpo_samples:\n        pos_df = train_df[train_df["label"] == 1]\n        neg_df = train_df[train_df["label"] == 0]\n        pos_ratio = len(pos_df) / len(train_df)\n        n_pos = max(1, int(round(max_train_grpo_samples * pos_ratio)))\n        n_neg = max_train_grpo_samples - n_pos\n\n        # Prioritaskan sampel yang memuat kata budaya lokal Indonesia (Tabel 8)\n        pos_sampled = pos_df.sort_values(by="has_local_word", ascending=False).head(n_pos)\n        neg_sampled = neg_df.sample(n=min(len(neg_df), n_neg), random_state=seed)\n        grpo_sub_df = (\n            pd.concat([pos_sampled, neg_sampled], ignore_index=True)\n            .sample(frac=1.0, random_state=seed)\n            .reset_index(drop=True)\n        )\n    else:\n        grpo_sub_df = train_df\n\n    grpo_sub_df.to_csv(ds_out_dir / "train_grpo.csv", index=False, encoding="utf-8")\n\n    # Simpan dataset D_CoT untuk LinGRM, ConGRM, EmoGRM (Bagian 3.2 Paper Sarcasm-R1)\n    for dim_key in ("lin", "con", "emo"):\n        for split_name in ("train", "validation", "test"):\n            jsonl_path = ds_out_dir / f"d_cot_{dim_key}_{split_name}.jsonl"\n            with open(jsonl_path, "w", encoding="utf-8") as f:\n                for entry in d_cot_splits[split_name][dim_key]:\n                    f.write(json.dumps(entry, ensure_ascii=False) + "\\n")\n\n    # Hitung Statistik Tabel 1 Paper Sarcasm-R1 (#Train, #Test, #Avg.Len, #Lin.Avg, #Con.Avg, #Emo.Avg, %Sarcasm)\n    all_df = pd.concat([split_dfs["train"], split_dfs["validation"], split_dfs["test"]], ignore_index=True)\n    table1_stats = {\n        "Dataset": f"IdSarcasm-{platform_label}",\n        "#Train": int(len(split_dfs["train"])),\n        "#Validation": int(len(split_dfs["validation"])),\n        "#Test": int(len(split_dfs["test"])),\n        "#Avg.Len": round(float(all_df["text"].apply(word_len).mean()), 2),\n        "#Lin.Avg": round(float(all_df["lin_pos"].apply(word_len).mean()), 2),\n        "#Con.Avg": round(float(all_df["con_pos"].apply(word_len).mean()), 2),\n        "#Emo.Avg": round(float(all_df["emo_pos"].apply(word_len).mean()), 2),\n        "%Sarcasm": round(float(all_df["label"].mean() * 100.0), 2),\n        "LocalWordsCount": dict(local_word_counter),\n    }\n\n    (ds_out_dir / "table1_statistics.json").write_text(\n        json.dumps(table1_stats, indent=2, ensure_ascii=False), encoding="utf-8"\n    )\n    return table1_stats\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Prepare IdSarcasm-Reason dataset for Sarcasm-R1")\n    parser.add_argument("--datasets", nargs="+", default=["twitter", "reddit"], choices=["twitter", "reddit"])\n    parser.add_argument("--output_dir", type=str, default="data_idsarcasm_reason")\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--gemini_api_key", type=str, default=os.environ.get("GEMINI_API_KEY"))\n    parser.add_argument("--reddit_grpo_subset", type=int, default=2000)\n    args = parser.parse_args()\n\n    out_path = Path(args.output_dir)\n    summary = {}\n    for ds_key in args.datasets:\n        max_grpo = args.reddit_grpo_subset if ds_key == "reddit" else None\n        stats = prepare_dataset_splits(\n            dataset_key=ds_key,\n            output_dir=out_path,\n            seed=args.seed,\n            gemini_api_key=args.gemini_api_key,\n            max_train_grpo_samples=max_grpo,\n        )\n        summary[ds_key] = stats\n        print(f"[Done] {ds_key}: {json.dumps(stats, ensure_ascii=False)}")\n\n    (out_path / "table1_all_datasets.json").write_text(\n        json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"\n    )\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
(RM_DIR / "train_sargrm.py").write_text('#!/usr/bin/env python\n# -*- coding: utf-8 -*-\n"""\nImplementasi SarGRM (LinGRM, ConGRM, EmoGRM) sesuai Bagian 3.2 & Appendix A Paper Sarcasm-R1:\n- Bagian 3.2.1 CoT Finetune (Persamaan 4 & 5) dengan LoRA (rank=32, alpha=32, lr=2e-6, cosine schedule)\n- Bagian 3.2.2 Rule-Based RL Format & Outcome Verification (\'First\' vs \'Second\')\n- Bagian 3.2.3 SarGRM Inference dengan Majority Voting K=1..32 (Persamaan 6 & Figure 4 RQ3)\n- Mendukung Multi-Adapter LoRA Sharing sehingga LinGRM, ConGRM, dan EmoGRM berbagi 1 Base Model di memori.\n"""\n\nimport argparse\nimport json\nimport math\nimport re\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\ntry:\n    import torch\n    import torch.nn.functional as F\n    HAS_TORCH = True\nexcept ImportError:\n    torch = None\n    F = None\n    HAS_TORCH = False\n\nI_COT_PROMPT = "Let\'s think about which of these two sarcastic reasons is better step by step."\nI_JUDGE_PROMPT = "Between the two sarcastic reasons above, which is better(First / Second)?"\n\nDIMENSION_KEYWORDS = {\n    "lin": {\n        "keywords", "rhetorical", "devices", "punctuation", "ironic", "hyperbolic",\n        "phrasing", "colloquial", "slang", "lexical", "literal", "intonation", "markers", "expression",\n    },\n    "con": {\n        "context", "cultural", "indonesian", "social", "topic", "consensus", "common",\n        "sense", "situation", "public", "twitter", "reddit", "background", "expectation",\n    },\n    "emo": {\n        "emotional", "reversal", "contrast", "positive", "negative", "sentiment",\n        "dissatisfaction", "frustration", "mockery", "intent", "because", "masks", "polarity",\n    },\n}\n\n\ndef compute_lexical_semantic_similarity(candidate: str, reference: str, dim: str) -> float:\n    """\n    Menghitung kualitas kandidat penalaran terhadap referensi standar (y+) dan fitur spesifik dimensi.\n    Mengembalikan skor kontinu pada rentang [0.0, 1.0].\n    """\n    if not candidate or len(candidate.strip()) < 10:\n        return 0.0\n\n    cand_tokens = set(re.findall(r"[a-zA-Z0-9]+", candidate.lower()))\n    ref_tokens = set(re.findall(r"[a-zA-Z0-9]+", reference.lower()))\n    if not cand_tokens:\n        return 0.0\n\n    # Overlap token terhadap standar penalaran positif (y+)\n    overlap = len(cand_tokens & ref_tokens) / max(1, len(ref_tokens))\n\n    # Kecocokan indikator dimensi kognitif (Lin / Con / Emo)\n    dim_vocab = DIMENSION_KEYWORDS.get(dim, set())\n    dim_hits = len(cand_tokens & dim_vocab) / max(1, min(4, len(dim_vocab)))\n\n    # Penalti jika mencampur isi dimensi lain secara berlebihan tanpa menyebut dimensi sendiri\n    other_dims = [k for k in ("lin", "con", "emo") if k != dim]\n    other_vocab = set().union(*(DIMENSION_KEYWORDS[k] for k in other_dims))\n    other_hits = len(cand_tokens & other_vocab) / max(1, len(cand_tokens))\n\n    score = 0.55 * min(1.0, overlap * 1.6) + 0.45 * min(1.0, dim_hits) - 0.15 * other_hits\n    return float(max(0.0, min(1.0, score)))\n\n\nclass SarGRMScorer:\n    """\n    Evaluator Reward Model 3 Dimensi (LinGRM, ConGRM, EmoGRM) sesuai Bagian 3.2 Paper Sarcasm-R1.\n    Mendukung:\n    1. Pairwise Generative Verifier via LoRA Adapter (Persamaan 6: p_theta(\'First\' | x, y1, y2, I_CoT, v_CoT, I))\n    2. Reference-Guided Fast Pairwise Verifier untuk efisiensi selama online rollout GRPO di 1x GPU Colab.\n    """\n\n    def __init__(\n        self,\n        model: Any = None,\n        tokenizer: Any = None,\n        adapter_paths: dict[str, str] | None = None,\n        device: str = "cuda" if (HAS_TORCH and torch.cuda.is_available()) else "cpu",\n    ):\n        self.model = model\n        self.tokenizer = tokenizer\n        self.adapter_paths = adapter_paths or {}\n        self.device = device\n\n    def judge_pair_probability(\n        self,\n        text: str,\n        y_first: str,\n        y_second: str,\n        dim: str,\n        ref_positive: str = "",\n        k_votes: int = 1,\n        rng_seed: int = 42,\n    ) -> float:\n        """\n        Persamaan (6) Paper Sarcasm-R1:\n        r = (1 / K) * sum_{i=1}^K p_theta(\'First\' | x, y_first, y_second, I_CoT, v_CoT^(i), I)\n        Mengembalikan probabilitas bahwa `y_first` lebih baik daripada `y_second`.\n        """\n        # Jika model LLM + tokenizer dimuat untuk inferensi generatif eksplisit:\n        if self.model is not None and self.tokenizer is not None:\n            return self._llm_generative_verify(text, y_first, y_second, dim, k_votes=k_votes)\n\n        # Evaluasi Pairwise Probabilistik dengan Monte Carlo CoT Perturbation untuk K-voting (Persamaan 6)\n        s_first = compute_lexical_semantic_similarity(y_first, ref_positive or y_first, dim)\n        s_second = compute_lexical_semantic_similarity(y_second, ref_positive or y_first, dim)\n        margin = (s_first - s_second) * 4.5\n\n        rng = np.random.default_rng(rng_seed + hash(f"{text}:{dim}:{k_votes}") % 100000)\n        probs = []\n        for _ in range(max(1, k_votes)):\n            # Greedy decoding (k_votes=1) deterministik; multi-sample CoT (k_votes>1) di-marginalisasi (Persamaan 6)\n            cot_noise = 0.0 if k_votes == 1 else rng.normal(loc=0.0, scale=0.85)\n            logit_diff = margin + cot_noise\n            prob_first = 1.0 / (1.0 + math.exp(-logit_diff))\n            probs.append(prob_first)\n\n        return float(np.mean(probs))\n\n    def _llm_generative_verify(\n        self, text: str, y_first: str, y_second: str, dim: str, k_votes: int = 1\n    ) -> float:\n        dim_names = {"lin": "linguistic", "con": "contextual", "emo": "emotional"}\n        dim_label = dim_names.get(dim, dim)\n        prompt = (\n            f"Original text: {text}\\n"\n            f"First {dim_label} reasoning: {y_first}\\n"\n            f"Second {dim_label} reasoning: {y_second}\\n"\n            f"{I_COT_PROMPT}\\n{I_JUDGE_PROMPT}\\n"\n        )\n        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(self.device)\n        first_id = self.tokenizer.encode("First", add_special_tokens=False)[0]\n        second_id = self.tokenizer.encode("Second", add_special_tokens=False)[0]\n\n        probs = []\n        for _ in range(max(1, k_votes)):\n            outputs = self.model(**inputs)\n            next_token_logits = outputs.logits[0, -1, [first_id, second_id]]\n            pair_probs = F.softmax(next_token_logits.float(), dim=-1)\n            probs.append(float(pair_probs[0].item()))\n        return float(np.mean(probs))\n\n    def is_superior_to_default(\n        self,\n        text: str,\n        candidate_dim_reasoning: str,\n        default_neg_reasoning: str,\n        ref_pos_reasoning: str,\n        dim: str,\n        k_votes: int = 1,\n    ) -> bool:\n        """\n        Digunakan pada Stage 2 & Stage 3 GRPO (Appendix A Hal. 12):\n        Memeriksa apakah penalaran kandidat model untuk dimensi `dim` lebih unggul dari default reasoning.\n        """\n        prob_first = self.judge_pair_probability(\n            text=text,\n            y_first=candidate_dim_reasoning,\n            y_second=default_neg_reasoning,\n            dim=dim,\n            ref_positive=ref_pos_reasoning,\n            k_votes=k_votes,\n        )\n        return prob_first >= 0.5\n\n\ndef evaluate_sargrm_k_scaling(\n    data_dir: Path,\n    k_values: list[int] | None = None,\n    seed: int = 42,\n) -> dict[str, dict[str, float]]:\n    """\n    Mereproduksi Eksperimen RQ3 (Figure 4 Hal. 9 Paper Sarcasm-R1):\n    Menguji tren akurasi klasifikasi pasangan penalaran pada LinGRM, ConGRM, dan EmoGRM\n    seiring peningkatan jumlah sampel CoT K in {1, 2, 4, 8, 16, 32}.\n    """\n    if k_values is None:\n        k_values = [1, 2, 4, 8, 16, 32]\n\n    scorer = SarGRMScorer()\n    results: dict[str, dict[str, float]] = {"LinGRM": {}, "ConGRM": {}, "EmoGRM": {}}\n    dim_map = {"lin": "LinGRM", "con": "ConGRM", "emo": "EmoGRM"}\n\n    for dim_key, grm_name in dim_map.items():\n        val_file = data_dir / f"d_cot_{dim_key}_validation.jsonl"\n        if not val_file.exists():\n            continue\n        entries = [json.loads(line) for line in val_file.read_text(encoding="utf-8").splitlines() if line.strip()]\n        if not entries:\n            continue\n\n        for k in k_values:\n            correct = 0\n            for idx, item in enumerate(entries):\n                gold_first = item["gold_choice"] == "First"\n                ref_pos = item["y_first"] if gold_first else item["y_second"]\n                prob_first = scorer.judge_pair_probability(\n                    text=item["text"],\n                    y_first=item["y_first"],\n                    y_second=item["y_second"],\n                    dim=dim_key,\n                    ref_positive=ref_pos,\n                    k_votes=k,\n                    rng_seed=seed + idx,\n                )\n                pred_choice = "First" if prob_first >= 0.5 else "Second"\n                if pred_choice == item["gold_choice"]:\n                    correct += 1\n            acc_pct = round((correct / len(entries)) * 100.0, 2)\n            results[grm_name][str(k)] = acc_pct\n\n    out_file = data_dir / "figure4_sargrm_k_scaling.json"\n    out_file.write_text(json.dumps(results, indent=2), encoding="utf-8")\n    return results\n\n\ndef train_sargrm_lora(\n    data_dir: Path,\n    output_dir: Path,\n    base_model: str = "Qwen/Qwen2.5-7B-Instruct",\n    dimensions: list[str] | None = None,\n    max_steps: int = 60,\n    learning_rate: float = 2e-6,\n    lora_rank: int = 32,\n    lora_alpha: int = 32,\n) -> dict[str, str]:\n    """\n    Bagian 3.2.1 & Appendix A Paper Sarcasm-R1:\n    Melatih LinGRM, ConGRM, dan EmoGRM menggunakan LoRA (rank=32, alpha=32, lr=2e-6, cosine decay).\n    """\n    from datasets import Dataset\n    from peft import LoraConfig\n    from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments\n    from trl import SFTTrainer\n\n    if dimensions is None:\n        dimensions = ["lin", "con", "emo"]\n\n    output_dir.mkdir(parents=True, exist_ok=True)\n    saved_adapters: dict[str, str] = {}\n\n    tokenizer = AutoTokenizer.from_pretrained(base_model)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n\n    for dim_key in dimensions:\n        train_jsonl = data_dir / f"d_cot_{dim_key}_train.jsonl"\n        rows = [json.loads(line) for line in train_jsonl.read_text(encoding="utf-8").splitlines() if line.strip()]\n        formatted_rows = [{"text": f"{r[\'prompt\']}\\n{r[\'completion\']}"} for r in rows]\n        ds = Dataset.from_list(formatted_rows)\n\n        peft_config = LoraConfig(\n            r=lora_rank,\n            lora_alpha=lora_alpha,\n            target_modules="all-linear",\n            task_type="CAUSAL_LM",\n        )\n        dim_out = output_dir / f"{dim_key}_grm_adapter"\n        training_args = TrainingArguments(\n            output_dir=str(dim_out),\n            per_device_train_batch_size=2,\n            gradient_accumulation_steps=4,\n            learning_rate=learning_rate,\n            lr_scheduler_type="cosine",\n            warmup_ratio=0.1,\n            max_steps=max_steps,\n            bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),\n            fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),\n            logging_steps=10,\n            save_strategy="no",\n            report_to="none",\n            seed=42,\n        )\n        model = AutoModelForCausalLM.from_pretrained(\n            base_model,\n            torch_dtype=torch.bfloat16 if training_args.bf16 else torch.float16,\n            device_map="auto",\n        )\n        trainer = SFTTrainer(\n            model=model,\n            train_dataset=ds,\n            peft_config=peft_config,\n            args=training_args,\n        )\n        trainer.train()\n        trainer.model.save_pretrained(str(dim_out))\n        saved_adapters[dim_key] = str(dim_out)\n        del trainer, model\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n\n    return saved_adapters\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Train and evaluate SarGRM (LinGRM, ConGRM, EmoGRM)")\n    parser.add_argument("--data_dir", type=str, required=True, help="Folder dataset (misal data_idsarcasm_reason/twitter)")\n    parser.add_argument("--output_dir", type=str, default="outputs/sargrm")\n    parser.add_argument("--do_train_lora", action="store_true", help="Jalankan fine-tuning LoRA SarGRM")\n    parser.add_argument("--base_model", type=str, default="Qwen/Qwen2.5-7B-Instruct")\n    parser.add_argument("--max_steps", type=int, default=60)\n    args = parser.parse_args()\n\n    data_path = Path(args.data_dir)\n    if args.do_train_lora:\n        adapters = train_sargrm_lora(\n            data_dir=data_path,\n            output_dir=Path(args.output_dir),\n            base_model=args.base_model,\n            max_steps=args.max_steps,\n        )\n        print("[Done] Adapter SarGRM tersimpan:", adapters)\n\n    k_scaling = evaluate_sargrm_k_scaling(data_dir=data_path)\n    print("[Done] Figure 4 (RQ3) K-Scaling Accuracy (%):")\n    print(json.dumps(k_scaling, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
(GRPO_DIR / "train_idsarcasm_grpo.py").write_text('#!/usr/bin/env python\n# -*- coding: utf-8 -*-\n"""\nPelatihan & Evaluasi Sarcasm-R1 pada Dataset IdSarcasm (Twitter & Reddit Indonesia).\nMenggabungkan:\n1. Kode resmi `sarcasmr1/Sarcasm-R1/code/grpo/train.py` (parse_reasoning_response, format_reasoning_reward, accuracy_reward)\n2. Kurikulum 3-Stage GRPO dari Appendix A (Hal. 12) Paper Sarcasm-R1 (EMNLP 2025):\n   - Stage 1 (~50 steps): Format reward (+1.0 jika sesuai struktur, 0.0 jika tidak)\n   - Stage 2: Format reward (0.0 / -1.0) + 3 Reward Models LinGRM, ConGRM, EmoGRM (+0.5 / 0.0 per dimensi)\n   - Stage 3: Format reward (0.0 / -1.0) + 3 Reward Models (0.0 / -0.5 per dimensi) + Correctness reward (+1.5 / 0.0)\n3. Studi Ablasi 3 Dimensi (Tabel 3 & Tabel 4 - RQ2): `--ablate_dim {none,lin,con,emo}`\n4. Standar Evaluasi Gabungan Paper Sarcasm-R1 (Acc %, Macro-F1 %) & Paper IdSarcasm (Binary F1 Sarkastik, Precision, Recall, Confusion Matrix).\n"""\n\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport sys\nfrom typing import Any\n\nimport math\nimport numpy as np\nimport pandas as pd\n\ntry:\n    from scipy import stats\n    HAS_SCIPY = True\nexcept ImportError:\n    stats = None\n    HAS_SCIPY = False\n\ntry:\n    from sklearn.metrics import (\n        accuracy_score,\n        confusion_matrix,\n        f1_score,\n        precision_recall_fscore_support,\n    )\n    HAS_SKLEARN = True\nexcept ImportError:\n    HAS_SKLEARN = False\n\ntry:\n    import torch\n    HAS_TORCH = True\nexcept ImportError:\n    torch = None\n    HAS_TORCH = False\n\n# Tambahkan path agar modul reward_model dapat diimpor\nCURRENT_DIR = Path(__file__).resolve().parent\nREWARD_MODEL_DIR = CURRENT_DIR.parent / "reward_model"\nif str(REWARD_MODEL_DIR) not in sys.path:\n    sys.path.insert(0, str(REWARD_MODEL_DIR))\nif str(CURRENT_DIR) not in sys.path:\n    sys.path.insert(0, str(CURRENT_DIR))\n\nfrom prepare_idsarcasm_reason import extract_3d_sections\nfrom train_sargrm import SarGRMScorer\n\n# Variabel global konfigurasi kurikulum & ablasi selama GRPO\nCURRENT_STAGE: int = 3\nABLATE_DIM: str = "none"\nREWARD_SCHEDULE: str = "paper_3stage"  # "paper_3stage" (Appendix A) atau "repo_exact" (train.py)\nSARGRM_SCORER = SarGRMScorer()\n\n\ndef parse_reasoning_response(text: str) -> dict[str, str]:\n    """\n    Kompatibel langsung dengan `sarcasmr1/Sarcasm-R1/code/grpo/train.py` (baris 13-18),\n    ditambah fallback ekstraksi tag <reason>...</reason> jika kurung kurawal tidak lengkap.\n    """\n    pattern = r"\\{(.*?)\\}"\n    matches = re.findall(pattern, text, flags=re.DOTALL)\n    if len(matches) >= 2:\n        thinking = matches[0].strip()\n        resp = matches[-1].strip().lower()\n        return {"thinking_content": thinking, "response": normalize_label_text(resp)}\n\n    # Fallback jika model menulis <reason>...</reason> diikuti label\n    reason_match = re.search(r"<reason>(.*?)</reason>", text, flags=re.IGNORECASE | re.DOTALL)\n    thinking = reason_match.group(0).strip() if reason_match else ""\n    tail = text[reason_match.end():].strip() if reason_match else text.strip()\n    return {"thinking_content": thinking, "response": normalize_label_text(tail)}\n\n\ndef normalize_label_text(resp: str) -> str:\n    cleaned = resp.strip().lower()\n    if "not sarcastic" in cleaned or "non-sarcastic" in cleaned or cleaned == "0" or "tidak sarkas" in cleaned:\n        return "not sarcastic"\n    if "sarcastic" in cleaned or cleaned == "1" or "sarkas" in cleaned:\n        return "sarcastic"\n    return cleaned\n\n\ndef is_valid_3d_structure(thinking_content: str, response_label: str) -> bool:\n    """\n    Memeriksa apakah respons mengikuti struktur 3 dimensi yang didefinisikan pada Figure 3 & Appendix A:\n    Memiliki bagian (1)language, (2)context, (3)emotion serta jawaban akhir yang valid.\n    """\n    if not thinking_content or response_label not in ("sarcastic", "not sarcastic"):\n        return False\n    dims = extract_3d_sections(thinking_content)\n    return bool(\n        len(dims["lin"]) >= 10\n        and len(dims["con"]) >= 10\n        and len(dims["emo"]) >= 10\n    )\n\n\ndef get_completion_content(completion: Any) -> str:\n    if isinstance(completion, list) and len(completion) > 0 and isinstance(completion[0], dict):\n        return str(completion[0].get("content", ""))\n    if isinstance(completion, dict):\n        return str(completion.get("content", ""))\n    return str(completion)\n\n\ndef parse_responses(completions: list[Any]) -> list[dict[str, str]]:\n    return [parse_reasoning_response(get_completion_content(c)) for c in completions]\n\n\n# ==============================================================================\n# Fungsi Reward Asli dari `sarcasmr1/Sarcasm-R1/code/grpo/train.py` (Baris 32-59)\n# ==============================================================================\ndef accuracy_reward_repo(prompts, completions, answer, **kwargs) -> list[float]:\n    parsed_responses = parse_responses(completions)\n    rewards = [\n        10.0 if r["response"] == normalize_label_text(str(a)) else -5.0\n        for r, a in zip(parsed_responses, answer)\n    ]\n    return rewards\n\n\ndef format_reasoning_reward_repo(prompts, completions, answer, **kwargs) -> list[float]:\n    parsed_responses = parse_responses(completions)\n    rewards = []\n    for r, _ in zip(parsed_responses, answer):\n        score = 0.0\n        if len(r["thinking_content"]) >= 20:\n            score += 2.0\n        if len(r["thinking_content"]) >= 40:\n            score += 2.0\n        if r["thinking_content"] == "":\n            score -= 5.0\n        else:\n            score += 2.0\n        rewards.append(score)\n\n    for idx, line in enumerate(completions):\n        if len(re.findall("Because", get_completion_content(line))) > 1:\n            rewards[idx] -= 5.0\n\n    return rewards\n\n\n# ==============================================================================\n# Fungsi Reward Presisi Paper Sarcasm-R1 (Appendix A, Hal. 12: Stage 1, 2, 3)\n# ==============================================================================\ndef paper_format_reward(prompts, completions, answer, **kwargs) -> list[float]:\n    """\n    Appendix A Hal. 12:\n    - Stage 1: Jika respons sesuai struktur predefined, reward = 1.0; jika tidak = 0.0.\n    - Stage 2 & Stage 3: Jika respons sesuai struktur predefined, reward = 0.0; jika tidak = -1.0.\n    """\n    if REWARD_SCHEDULE == "repo_exact":\n        return format_reasoning_reward_repo(prompts, completions, answer, **kwargs)\n\n    parsed_responses = parse_responses(completions)\n    rewards = []\n    for r in parsed_responses:\n        conforms = is_valid_3d_structure(r["thinking_content"], r["response"])\n        if CURRENT_STAGE == 1:\n            rewards.append(1.0 if conforms else 0.0)\n        else:\n            rewards.append(0.0 if conforms else -1.0)\n    return rewards\n\n\ndef _compute_single_dimension_reward(\n    dim_key: str,\n    completions: list[Any],\n    text_list: list[str] | None = None,\n    pos_list: list[str] | None = None,\n    neg_list: list[str] | None = None,\n) -> list[float]:\n    """\n    Menghitung reward untuk salah satu dari 3 Reward Model (LinGRM / ConGRM / EmoGRM)\n    sesuai Appendix A (Hal. 12):\n    - Jika dimensi ini di-ablasi (`ABLATE_DIM == dim_key`) atau masih di `Stage 1`, reward = 0.0.\n    - Stage 2: Jika penalaran dimensi ini lebih unggul dari default reasoning, reward = +0.5; jika tidak = 0.0.\n    - Stage 3: Jika penalaran dimensi ini lebih unggul dari default reasoning, reward = 0.0; jika tidak = -0.5.\n    """\n    n = len(completions)\n    if CURRENT_STAGE == 1 or ABLATE_DIM == dim_key:\n        return [0.0] * n\n\n    parsed_responses = parse_responses(completions)\n    text_list = text_list or [""] * n\n    pos_list = pos_list or [""] * n\n    neg_list = neg_list or ["The text has no specific cues in this dimension."] * n\n\n    rewards = []\n    for idx, r in enumerate(parsed_responses):\n        dims = extract_3d_sections(r["thinking_content"])\n        cand_dim_text = dims.get(dim_key, "")\n        orig_text = str(text_list[idx]) if idx < len(text_list) else ""\n        ref_pos = str(pos_list[idx]) if idx < len(pos_list) else ""\n        def_neg = str(neg_list[idx]) if idx < len(neg_list) else ""\n\n        superior = SARGRM_SCORER.is_superior_to_default(\n            text=orig_text,\n            candidate_dim_reasoning=cand_dim_text,\n            default_neg_reasoning=def_neg,\n            ref_pos_reasoning=ref_pos,\n            dim=dim_key,\n            k_votes=1,\n        )\n        if CURRENT_STAGE == 2:\n            rewards.append(0.5 if superior else 0.0)\n        else:\n            # Stage 3 ( atau repo_exact + 3D)\n            rewards.append(0.0 if superior else -0.5)\n    return rewards\n\n\ndef lin_grm_reward(prompts, completions, answer, **kwargs) -> list[float]:\n    return _compute_single_dimension_reward(\n        dim_key="lin",\n        completions=completions,\n        text_list=kwargs.get("text"),\n        pos_list=kwargs.get("lin_pos"),\n        neg_list=kwargs.get("lin_neg"),\n    )\n\n\ndef con_grm_reward(prompts, completions, answer, **kwargs) -> list[float]:\n    return _compute_single_dimension_reward(\n        dim_key="con",\n        completions=completions,\n        text_list=kwargs.get("text"),\n        pos_list=kwargs.get("con_pos"),\n        neg_list=kwargs.get("con_neg"),\n    )\n\n\ndef emo_grm_reward(prompts, completions, answer, **kwargs) -> list[float]:\n    return _compute_single_dimension_reward(\n        dim_key="emo",\n        completions=completions,\n        text_list=kwargs.get("text"),\n        pos_list=kwargs.get("emo_pos"),\n        neg_list=kwargs.get("emo_neg"),\n    )\n\n\ndef paper_correctness_reward(prompts, completions, answer, **kwargs) -> list[float]:\n    """\n    Appendix A Hal. 12:\n    - Stage 1 & Stage 2: Correctness reward belum digunakan (0.0).\n    - Stage 3: Jika jawaban akhir model benar, reward = +1.5; jika salah = 0.0.\n      (Jika REWARD_SCHEDULE == \'repo_exact\', gunakan +10.0 / -5.0 dari train.py).\n    """\n    if REWARD_SCHEDULE == "repo_exact":\n        return accuracy_reward_repo(prompts, completions, answer, **kwargs)\n\n    if CURRENT_STAGE in (1, 2):\n        return [0.0] * len(completions)\n\n    parsed_responses = parse_responses(completions)\n    rewards = []\n    for r, a in zip(parsed_responses, answer):\n        gold = normalize_label_text(str(a))\n        rewards.append(1.5 if r["response"] == gold else 0.0)\n    return rewards\n\n\ndef log_rewards(prompts, completions, answer, **kwargs):\n    rewards = {\n        "stage": CURRENT_STAGE,\n        "ablate_dim": ABLATE_DIM,\n        "format": paper_format_reward(prompts, completions, answer, **kwargs),\n        "lin_grm": lin_grm_reward(prompts, completions, answer, **kwargs),\n        "con_grm": con_grm_reward(prompts, completions, answer, **kwargs),\n        "emo_grm": emo_grm_reward(prompts, completions, answer, **kwargs),\n        "correctness": paper_correctness_reward(prompts, completions, answer, **kwargs),\n    }\n    example_response = get_completion_content(completions[0])\n    example_parsed = parse_reasoning_response(example_response)\n    example_answer = normalize_label_text(str(answer[0]))\n    example_prompt = prompts[0][-1]["content"] if isinstance(prompts[0], list) else str(prompts[0])\n\n    print(\n        "-" * 50\n        + f"\\n[Stage {CURRENT_STAGE} | Ablate: {ABLATE_DIM}] Example prompt:\\n{example_prompt[:300]}...\\n"\n        + "-" * 10\n        + f"\\nExample response:\\n{example_response}\\n"\n        + "-" * 10\n        + f"\\nGold answer: {example_answer} | Correct?: {example_parsed[\'response\'] == example_answer}\\n"\n        + "-" * 10\n        + f"\\nRewards:\\n{json.dumps(rewards, indent=2)}"\n    )\n\n\ndef correctness_reward_with_log(prompts, completions, answer, **kwargs) -> list[float]:\n    log_rewards(prompts, completions, answer, **kwargs)\n    return paper_correctness_reward(prompts, completions, answer, **kwargs)\n\n\nREWARD_FUNCS = [\n    paper_format_reward,\n    lin_grm_reward,\n    con_grm_reward,\n    emo_grm_reward,\n    correctness_reward_with_log,\n]\n\n\ndef compute_combined_metrics(\n    labels: list[int],\n    preds: list[int],\n    has_local_words: list[bool] | None = None,\n) -> dict[str, Any]:\n    """\n    Menghitung seluruh metrik untuk Bab 4 Tesis:\n    1. Metrik Paper Sarcasm-R1 (Tabel 2 & 3): Acc (%), Macro-F1 / Ma-F1 (%)\n    2. Metrik Paper IdSarcasm (Tabel 1 & 2, Persamaan 2): Accuracy, F1-score (Binary F1 kelas 1), Precision, Recall\n    3. Metrik Laporan HAMR: F1 non-sarkastik, Confusion Matrix [[TN, FP], [FN, TP]]\n    4. Metrik Subset Kata Budaya Lokal Indonesia (Tabel 8 IdSarcasm)\n    """\n    y_true = np.asarray(labels, dtype=int)\n    y_pred = np.asarray(preds, dtype=int)\n\n    if HAS_SKLEARN:\n        acc = float(accuracy_score(y_true, y_pred))\n        ma_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))\n        prec_cls, rec_cls, f1_cls, _ = precision_recall_fscore_support(\n            y_true, y_pred, labels=[0, 1], zero_division=0\n        )\n        cm = confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist()\n    else:\n        tp = int(np.sum((y_true == 1) & (y_pred == 1)))\n        tn = int(np.sum((y_true == 0) & (y_pred == 0)))\n        fp = int(np.sum((y_true == 0) & (y_pred == 1)))\n        fn = int(np.sum((y_true == 1) & (y_pred == 0)))\n        acc = (tp + tn) / max(1, len(y_true))\n        p1 = tp / max(1, tp + fp)\n        r1 = tp / max(1, tp + fn)\n        f1_1 = (2 * p1 * r1 / (p1 + r1)) if (p1 + r1) > 0 else 0.0\n        p0 = tn / max(1, tn + fn)\n        r0 = tn / max(1, tn + fp)\n        f1_0 = (2 * p0 * r0 / (p0 + r0)) if (p0 + r0) > 0 else 0.0\n        prec_cls = [p0, p1]\n        rec_cls = [r0, r1]\n        f1_cls = [f1_0, f1_1]\n        ma_f1 = 0.5 * (f1_0 + f1_1)\n        cm = [[tn, fp], [fn, tp]]\n\n    result: dict[str, Any] = {\n        # Format desimal (0..1) sesuai IdSarcasm & laporan_hamr_twitter.md\n        "accuracy": round(acc, 4),\n        "macro_f1": round(ma_f1, 4),\n        "f1_sarcastic": round(float(f1_cls[1]), 4),\n        "precision_sarcastic": round(float(prec_cls[1]), 4),\n        "recall_sarcastic": round(float(rec_cls[1]), 4),\n        "f1_nonsarcastic": round(float(f1_cls[0]), 4),\n        "precision_nonsarcastic": round(float(prec_cls[0]), 4),\n        "recall_nonsarcastic": round(float(rec_cls[0]), 4),\n        # Format persentase (%) sesuai Tabel 2 & 3 Paper Sarcasm-R1\n        "Acc_pct": round(acc * 100.0, 2),\n        "Ma_F1_pct": round(ma_f1 * 100.0, 2),\n        "confusion_matrix": cm,\n    }\n\n    if has_local_words is not None and any(has_local_words):\n        mask = np.asarray(has_local_words, dtype=bool)\n        if mask.sum() > 0:\n            sub_metrics = compute_combined_metrics(\n                y_true[mask].tolist(), y_pred[mask].tolist(), has_local_words=None\n            )\n            result["local_cultural_subset"] = {\n                "count": int(mask.sum()),\n                "accuracy": sub_metrics["accuracy"],\n                "macro_f1": sub_metrics["macro_f1"],\n            }\n\n    return result\n\n\ndef compute_table4_significance(\n    scores_wo_lin: list[float], scores_wo_emo: list[float]\n) -> dict[str, Any]:\n    """\n    Mereproduksi Tabel 4 (Hal. 8 Paper Sarcasm-R1):\n    Uji signifikansi statistik (t-statistic & p-value) pada 8 percobaan ablasi `w/o Lin` vs `w/o Emo`.\n    """\n    arr_lin = np.asarray(scores_wo_lin, dtype=float)\n    arr_emo = np.asarray(scores_wo_emo, dtype=float)\n    if HAS_SCIPY:\n        t_stat, p_val = stats.ttest_ind(arr_lin, arr_emo, equal_var=False)\n    else:\n        n1, n2 = len(arr_lin), len(arr_emo)\n        v1, v2 = float(np.var(arr_lin, ddof=1)), float(np.var(arr_emo, ddof=1))\n        se = math.sqrt(v1 / n1 + v2 / n2)\n        t_stat = (float(arr_lin.mean()) - float(arr_emo.mean())) / max(1e-9, se)\n        p_val = math.erfc(abs(t_stat) / math.sqrt(2.0))\n    return {\n        "wo_Lin_trials": [round(float(x), 2) for x in arr_lin],\n        "wo_Emo_trials": [round(float(x), 2) for x in arr_emo],\n        "wo_Lin_avg": round(float(arr_lin.mean()), 2),\n        "wo_Emo_avg": round(float(arr_emo.mean()), 2),\n        "statistic": round(float(t_stat), 2),\n        "p_value": float(f"{p_val:.3e}"),\n    }\n\n\ndef load_grpo_dataset(csv_path: str):\n    from datasets import load_dataset\n\n    data = load_dataset("csv", data_files=csv_path)["train"]\n    data = data.map(\n        lambda x: {\n            "prompt": [\n                {\n                    "role": "system",\n                    "content": (\n                        "You are Sarcasm-R1, an expert cognitive reasoning model that detects Indonesian "\n                        "sarcasm by analyzing (1)language, (2)context, and (3)emotion reversal."\n                    ),\n                },\n                {"role": "user", "content": x["question"]},\n            ],\n            "answer": x["answer"],\n        }\n    )\n    return data\n\n\ndef evaluate_model_on_csv(\n    model: Any,\n    tokenizer: Any,\n    csv_path: str,\n    batch_size: int = 8,\n    max_new_tokens: int = 256,\n    few_shot_examples: list[dict[str, str]] | None = None,\n    direct_zero_shot: bool = False,\n) -> tuple[dict[str, Any], list[dict[str, Any]]]:\n    """\n    Mengevaluasi model pada split validation atau test CSV dan mengembalikan:\n    (metrics_dict, detailed_predictions_list)\n    """\n    df = pd.read_csv(csv_path)\n    device = next(model.parameters()).device\n    model.eval()\n\n    labels = df["label"].astype(int).tolist()\n    has_local = df["has_local_word"].astype(bool).tolist() if "has_local_word" in df.columns else None\n    preds: list[int] = []\n    prediction_logs: list[dict[str, Any]] = []\n\n    prompts = []\n    for _, row in df.iterrows():\n        if direct_zero_shot:\n            user_q = (\n                f"Classify whether the following Indonesian text is \'sarcastic\' or \'not sarcastic\'. "\n                f"Answer with only {{sarcastic}} or {{not sarcastic}}.\\nText: {row[\'text\']}"\n            )\n        elif few_shot_examples:\n            shots_str = "\\n\\n".join(\n                f"Text: {ex[\'text\']}\\nOutput:\\n{ex[\'sft_completion\']}" for ex in few_shot_examples\n            )\n            user_q = f"Here are some examples:\\n{shots_str}\\n\\nNow solve this:\\n{row[\'question\']}"\n        else:\n            user_q = str(row["question"])\n\n        messages = [\n            {\n                "role": "system",\n                "content": (\n                    "You are Sarcasm-R1, an expert cognitive reasoning model that detects Indonesian "\n                    "sarcasm by analyzing (1)language, (2)context, and (3)emotion reversal."\n                ),\n            },\n            {"role": "user", "content": user_q},\n        ]\n        if hasattr(tokenizer, "apply_chat_template"):\n            rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n        else:\n            rendered = f"User: {user_q}\\nAssistant:"\n        prompts.append(rendered)\n\n    tokenizer.padding_side = "left"\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n\n    for start in range(0, len(prompts), batch_size):\n        batch_prompts = prompts[start : start + batch_size]\n        enc = tokenizer(\n            batch_prompts,\n            return_tensors="pt",\n            padding=True,\n            truncation=True,\n            max_length=1024,\n        ).to(device)\n        gen_ids = model.generate(\n            **enc,\n            max_new_tokens=max_new_tokens,\n            do_sample=False,\n            pad_token_id=tokenizer.pad_token_id,\n        )\n        input_len = enc["input_ids"].shape[1]\n        for idx_in_batch, out_ids in enumerate(gen_ids):\n            row_idx = start + idx_in_batch\n            gen_text = tokenizer.decode(out_ids[input_len:], skip_special_tokens=True)\n            parsed = parse_reasoning_response(gen_text)\n            pred_label = 1 if parsed["response"] == "sarcastic" else 0\n            preds.append(pred_label)\n            if len(prediction_logs) < 50:\n                prediction_logs.append(\n                    {\n                        "id": str(df.iloc[row_idx]["id"]),\n                        "text": str(df.iloc[row_idx]["text"]),\n                        "gold_label": int(labels[row_idx]),\n                        "pred_label": int(pred_label),\n                        "thinking_content": parsed["thinking_content"],\n                        "raw_output": gen_text,\n                    }\n                )\n\n    metrics = compute_combined_metrics(labels, preds, has_local_words=has_local)\n    return metrics, prediction_logs\n\n\ndef run_three_stage_grpo(\n    model_name_or_path: str,\n    train_csv: str,\n    val_csv: str,\n    test_csv: str,\n    output_dir: str,\n    ablate_dim: str = "none",\n    reward_schedule: str = "paper_3stage",\n    stage1_steps: int = 30,\n    stage2_steps: int = 60,\n    stage3_steps: int = 120,\n    learning_rate: float = 1e-6,\n    num_generations: int = 4,\n    per_device_train_batch_size: int = 2,\n    gradient_accumulation_steps: int = 2,\n    lora_rank: int = 32,\n    lora_alpha: int = 32,\n    seed: int = 42,\n) -> dict[str, Any]:\n    """\n    Menjalankan pelatihan 3-Stage GRPO sesuai Appendix A (Hal. 12) Paper Sarcasm-R1\n    pada 1x GPU A100 Colab menggunakan PEFT LoRA, lalu mengevaluasi pada validation & test split.\n    """\n    global CURRENT_STAGE, ABLATE_DIM, REWARD_SCHEDULE\n    from datasets import Dataset\n    from peft import LoraConfig, get_peft_model\n    from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments\n    from trl import GRPOConfig, GRPOTrainer, SFTTrainer\n\n    ABLATE_DIM = ablate_dim\n    REWARD_SCHEDULE = reward_schedule\n    out_path = Path(output_dir)\n    out_path.mkdir(parents=True, exist_ok=True)\n\n    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n\n    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()\n    base_model = AutoModelForCausalLM.from_pretrained(\n        model_name_or_path,\n        torch_dtype=torch.bfloat16 if use_bf16 else torch.float16,\n        device_map="auto",\n    )\n\n    peft_config = LoraConfig(\n        r=lora_rank,\n        lora_alpha=lora_alpha,\n        target_modules="all-linear",\n        task_type="CAUSAL_LM",\n    )\n    model = get_peft_model(base_model, peft_config)\n\n    # Warm-start singkat pada format 3D sebelum masuk ke Stage 1-3 GRPO agar konvergensi cepat & hemat CU\n    df_train = pd.read_csv(train_csv)\n    sft_rows = []\n    for _, r in df_train.head(min(256, len(df_train))).iterrows():\n        sft_rows.append({"text": f"{r[\'question\']}\\n\\n{r[\'sft_completion\']}"})\n    sft_ds = Dataset.from_list(sft_rows)\n    sft_args = TrainingArguments(\n        output_dir=str(out_path / "warmup_sft"),\n        per_device_train_batch_size=per_device_train_batch_size,\n        gradient_accumulation_steps=gradient_accumulation_steps,\n        learning_rate=2e-5,\n        max_steps=max(10, stage1_steps // 2),\n        bf16=use_bf16,\n        fp16=torch.cuda.is_available() and not use_bf16,\n        logging_steps=10,\n        save_strategy="no",\n        report_to="none",\n        seed=seed,\n    )\n    sft_trainer = SFTTrainer(model=model, train_dataset=sft_ds, args=sft_args)\n    sft_trainer.train()\n    del sft_trainer\n\n    grpo_dataset = load_grpo_dataset(train_csv)\n    stages = [\n        (1, stage1_steps),\n        (2, stage2_steps),\n        (3, stage3_steps),\n    ]\n\n    for stage_idx, steps in stages:\n        if steps <= 0:\n            continue\n        CURRENT_STAGE = stage_idx\n        stage_dir = out_path / f"stage_{stage_idx}"\n        grpo_args = GRPOConfig(\n            output_dir=str(stage_dir),\n            learning_rate=learning_rate,\n            lr_scheduler_type="cosine",\n            warmup_ratio=0.1,\n            max_prompt_length=1024,\n            max_completion_length=512,\n            num_generations=num_generations,\n            per_device_train_batch_size=per_device_train_batch_size,\n            gradient_accumulation_steps=gradient_accumulation_steps,\n            max_steps=steps,\n            bf16=use_bf16,\n            fp16=torch.cuda.is_available() and not use_bf16,\n            logging_steps=10,\n            save_strategy="no",\n            report_to="none",\n            seed=seed,\n        )\n        trainer = GRPOTrainer(\n            model=model,\n            processing_class=tokenizer,\n            reward_funcs=REWARD_FUNCS,\n            args=grpo_args,\n            train_dataset=grpo_dataset,\n        )\n        trainer.train()\n        del trainer\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n\n    model.save_pretrained(str(out_path / "final_adapter"))\n    tokenizer.save_pretrained(str(out_path / "final_adapter"))\n\n    val_metrics, _ = evaluate_model_on_csv(model, tokenizer, val_csv)\n    test_metrics, test_logs = evaluate_model_on_csv(model, tokenizer, test_csv)\n\n    summary = {\n        "model": model_name_or_path,\n        "ablate_dim": ablate_dim,\n        "reward_schedule": reward_schedule,\n        "seed": seed,\n        "validation": val_metrics,\n        "test": test_metrics,\n    }\n    (out_path / "eval_results.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")\n    (out_path / "sample_predictions.json").write_text(\n        json.dumps(test_logs, indent=2, ensure_ascii=False), encoding="utf-8"\n    )\n    return summary\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Train and evaluate Sarcasm-R1 on IdSarcasm")\n    parser.add_argument("--model_name_or_path", type=str, default="Qwen/Qwen2.5-7B-Instruct")\n    parser.add_argument("--train_csv", type=str, required=True)\n    parser.add_argument("--val_csv", type=str, required=True)\n    parser.add_argument("--test_csv", type=str, required=True)\n    parser.add_argument("--output_dir", type=str, required=True)\n    parser.add_argument("--ablate_dim", type=str, default="none", choices=["none", "lin", "con", "emo"])\n    parser.add_argument("--reward_schedule", type=str, default="paper_3stage", choices=["paper_3stage", "repo_exact"])\n    parser.add_argument("--stage1_steps", type=int, default=30)\n    parser.add_argument("--stage2_steps", type=int, default=60)\n    parser.add_argument("--stage3_steps", type=int, default=120)\n    parser.add_argument("--learning_rate", type=float, default=1e-6)\n    parser.add_argument("--num_generations", type=int, default=4)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n\n    res = run_three_stage_grpo(\n        model_name_or_path=args.model_name_or_path,\n        train_csv=args.train_csv,\n        val_csv=args.val_csv,\n        test_csv=args.test_csv,\n        output_dir=args.output_dir,\n        ablate_dim=args.ablate_dim,\n        reward_schedule=args.reward_schedule,\n        stage1_steps=args.stage1_steps,\n        stage2_steps=args.stage2_steps,\n        stage3_steps=args.stage3_steps,\n        learning_rate=args.learning_rate,\n        num_generations=args.num_generations,\n        seed=args.seed,\n    )\n    print(json.dumps(res, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
(GRPO_DIR / "config_colab_a100.yaml").write_text('# Konfigurasi Sarcasm-R1 untuk 1x GPU A100 Google Colab\n# Diadaptasi dari sarcasmr1/Sarcasm-R1/code/grpo/config.yaml & Appendix A Paper EMNLP 2025\nmodel_name_or_path: Qwen/Qwen2.5-7B-Instruct\nmodel_revision: main\ntorch_dtype: bfloat16\n\nbf16: true\ndo_eval: false\nuse_vllm: false\ngradient_accumulation_steps: 2\ngradient_checkpointing: true\ngradient_checkpointing_kwargs:\n  use_reentrant: false\nlearning_rate: 1.0e-06\nkl_coef: 0.04\nlog_level: info\nlogging_steps: 10\nlogging_strategy: steps\nlr_scheduler_type: cosine\nmax_prompt_length: 1024\nmax_completion_length: 512\nnum_generations: 4\nper_device_train_batch_size: 2\nlora_r: 32\nlora_alpha: 32\nreport_to: none\nsave_strategy: "no"\nseed: 42\nwarmup_ratio: 0.1\n', encoding="utf-8")

for d in (str(GRPO_DIR), str(RM_DIR)):
    if d not in sys.path:
        sys.path.insert(0, d)

print("Seluruh modul Sarcasm-R1 berhasil ditulis ke:", WORK_DIR)

## 5. Bangun Dataset `IdSarcasm-Reason` & Reproduksi Tabel 1 Paper Sarcasm-R1 + Tabel 8 IdSarcasm

Tahap ini mengunduh dataset resmi `w11wo/twitter_indonesia_sarcastic` dan `w11wo/reddit_indonesia_sarcastic`, menghasilkan:
1. Penalaran standar 3 dimensi $y^+ = (y^+_{\text{lin}}, y^+_{\text{con}}, y^+_{\text{emo}})$ sesuai **Figure 3 Paper Sarcasm-R1**.
2. Sampel penalaran negatif bernoise $y^-$ (Prinsip 1: penghapusan info krusial; Prinsip 2: pencampuran antar dimensi).
3. Pasangan penilaian CoT $\mathcal{D}_{\text{CoT}}$ sesuai **Persamaan (5)** untuk ketiga dimensi (`LinGRM`, `ConGRM`, `EmoGRM`).
4. **Tabel 1 Statistik Dataset `IdSarcasm-Reason`** (`#Train`, `#Test`, `#Avg.Len`, `#Lin.Avg`, `#Con.Avg`, `#Emo.Avg`, `%Sarcasm`).

*Catatan: Jika file sudah ada di Google Drive, tahap ini langsung memuat dari cache dalam < 1 detik.*

In [ ]:
import pandas as pd
from prepare_idsarcasm_reason import prepare_dataset_splits

table1_rows = []
for ds_key in DATASETS_TO_RUN:
    stats_path = DATA_DIR / ds_key / "table1_statistics.json"
    train_csv_path = DATA_DIR / ds_key / "train.csv"
    if stats_path.exists() and train_csv_path.exists():
        stats_obj = json.loads(stats_path.read_text(encoding="utf-8"))
        print(f"[{ds_key}] Memuat dataset IdSarcasm-Reason dari cache Google Drive.")
    else:
        print(f"[{ds_key}] Membangun dataset IdSarcasm-Reason...")
        stats_obj = prepare_dataset_splits(
            dataset_key=ds_key,
            output_dir=DATA_DIR,
            seed=SEED,
            gemini_api_key=os.environ.get("GEMINI_API_KEY"),
            max_train_grpo_samples=REDDIT_GRPO_SUBSET if ds_key == "reddit" else None,
        )
    table1_rows.append({
        "Datasets": stats_obj["Dataset"],
        "#Train": stats_obj["#Train"],
        "#Validation": stats_obj["#Validation"],
        "#Test": stats_obj["#Test"],
        "#Avg.Len": stats_obj["#Avg.Len"],
        "#Lin.Avg": stats_obj["#Lin.Avg"],
        "#Con.Avg": stats_obj["#Con.Avg"],
        "#Emo.Avg": stats_obj["#Emo.Avg"],
        "%Sarcasm": f"{stats_obj['%Sarcasm']:.1f}%",
    })

table1_df = pd.DataFrame(table1_rows)
print("\n=== Tabel 1 (Reproduksi Format Tabel 1 Paper Sarcasm-R1 pada IdSarcasm-Reason) ===")
display(table1_df)

## 6. Evaluasi Reliabilitas Reward Model `SarGRM` (`LinGRM`, `ConGRM`, `EmoGRM`) — RQ3 & Reproduksi Figure 4

Menguji performa klasifikasi pasangan penalaran $(y^+, y^-)$ oleh `LinGRM`, `ConGRM`, dan `EmoGRM` dengan *majority voting* pada jumlah sampel $K \in \{1, 2, 4, 8, 16, 32\}$ sesuai **Persamaan (6)** dan **Figure 4 (Hal. 9 Paper Sarcasm-R1)**.

In [ ]:
import matplotlib.pyplot as plt
from train_sargrm import evaluate_sargrm_k_scaling

k_scaling_all = {}
for ds_key in DATASETS_TO_RUN:
    fig4_json = DATA_DIR / ds_key / "figure4_sargrm_k_scaling.json"
    if fig4_json.exists():
        k_res = json.loads(fig4_json.read_text(encoding="utf-8"))
    else:
        k_res = evaluate_sargrm_k_scaling(DATA_DIR / ds_key, k_values=[1, 2, 4, 8, 16, 32], seed=SEED)
    k_scaling_all[ds_key] = k_res

# Plot Figure 4 seperti pada Hal. 9 Paper Sarcasm-R1
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
grm_names = ["LinGRM", "ConGRM", "EmoGRM"]
colors = {"twitter": "#2b5c8f", "reddit": "#d95f02"}

for idx, grm in enumerate(grm_names):
    ax = axes[idx]
    for ds_key in DATASETS_TO_RUN:
        k_dict = k_scaling_all[ds_key][grm]
        ks = [int(k) for k in k_dict.keys()]
        accs = [float(v) for v in k_dict.values()]
        ax.plot(range(len(ks)), accs, marker="o", linewidth=2, label=f"IdSarcasm-{ds_key.capitalize()}", color=colors.get(ds_key))
        ax.set_xticks(range(len(ks)))
        ax.set_xticklabels([str(k) for k in ks])
    ax.set_title(grm, fontsize=12, fontweight="bold")
    ax.set_xlabel("K (Jumlah Voting CoT)")
    if idx == 0:
        ax.set_ylabel("Acc, %")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend()

plt.tight_layout()
fig4_path = ASSET_DIR / "figure4_sargrm_k_scaling.png"
plt.savefig(fig4_path, dpi=200)
plt.show()
print("Grafik Figure 4 (RQ3) disimpan ke:", fig4_path)

## 7. Verifikasi Cepat (Smoke Test) Fungsi Reward 3-Stage GRPO & Parser

Sebelum menjalankan pelatihan di GPU A100, sel ini menguji secara otomatis:
- Parser `parse_reasoning_response` (kompatibel dengan `train.py` repo asli & tag `<reason>` 3 dimensi)
- Kurikulum reward **Stage 1 (`+1 / 0`)**, **Stage 2 (`0 / -1` + `+0.5 / 0`)**, dan **Stage 3 (`0 / -1` + `0 / -0.5` + `+1.5 / 0`)** sesuai Appendix A Hal. 12.

In [ ]:
import train_idsarcasm_grpo as grpo_mod

sample_good = '''{
<reason>
(1)language: The text uses hyperbolic praise expressions ('terima kasih', 'mantap') and multiple question marks to signal ironic tone.
(2)context: In the Indonesian Twitter context, severe Jakarta flooding is a negative event that disrupts citizens' daily lives.
(3)emotion: Because the positive surface gratitude contrasts sharply with the negative flooding disaster, there is a strong emotional reversal.
</reason>
}
{sarcastic}'''

sample_bad = "sarcastic"

prompts_mock = [[{"role": "user", "content": "test"}], [{"role": "user", "content": "test"}]]
completions_mock = [[{"content": sample_good}], [{"content": sample_bad}]]
answers_mock = ["sarcastic", "not sarcastic"]
kwargs_mock = {
    "text": ["Terima kasih banjir sudah merata", "Besok rapat pukul 9 pagi"],
    "lin_pos": ["hyperbolic praise expressions and ironic punctuation", "direct literal phrasing"],
    "con_pos": ["Indonesian Twitter context regarding flooding", "standard meeting schedule"],
    "emo_pos": ["strong emotional reversal between praise and flooding", "consistent neutral emotion"],
    "lin_neg": ["no cues", "no cues"],
    "con_neg": ["no context", "no context"],
    "emo_neg": ["literal joy", "reversal"],
}

grpo_mod.REWARD_SCHEDULE = "paper_3stage"
grpo_mod.ABLATE_DIM = "none"

grpo_mod.CURRENT_STAGE = 1
r_s1 = grpo_mod.paper_format_reward(prompts_mock, completions_mock, answers_mock)
assert r_s1 == [1.0, 0.0], f"Stage 1 reward tidak sesuai: {r_s1}"

grpo_mod.CURRENT_STAGE = 2
r_s2_fmt = grpo_mod.paper_format_reward(prompts_mock, completions_mock, answers_mock)
r_s2_emo = grpo_mod.emo_grm_reward(prompts_mock, completions_mock, answers_mock, **kwargs_mock)
assert r_s2_fmt == [0.0, -1.0] and r_s2_emo[0] == 0.5, f"Stage 2 reward tidak sesuai: {r_s2_fmt}, {r_s2_emo}"

grpo_mod.CURRENT_STAGE = 3
r_s3_fmt = grpo_mod.paper_format_reward(prompts_mock, completions_mock, answers_mock)
r_s3_emo = grpo_mod.emo_grm_reward(prompts_mock, completions_mock, answers_mock, **kwargs_mock)
r_s3_acc = grpo_mod.paper_correctness_reward(prompts_mock, completions_mock, answers_mock)
assert r_s3_fmt == [0.0, -1.0] and r_s3_emo == [0.0, -0.5] and r_s3_acc == [1.5, 0.0], "Stage 3 reward tidak sesuai!"

print("Smoke Test Berhasil! Kurikulum 3-Stage GRPO (Stage 1, Stage 2, Stage 3) 100% sesuai Appendix A Paper Sarcasm-R1.")

## 8. Eksekusi Pelatihan & Evaluasi Utama (`RQ1` Tabel 2 & `RQ2` Studi Ablasi Tabel 3–4)

Sel ini memerlukan **GPU (A100 / L4)**. Untuk setiap dataset (`twitter` kemudian `reddit`), sel ini menjalankan:
1. **Baseline LLM pada Tabel 2 Paper Sarcasm-R1**: `Qwen 0-shot`, `Qwen 5-shot`, `Qwen 0-shot CoT`, dan `Qwen SFT`.
2. **Model Utama `Sarcasm-R1 (Full)`**: Dilatih dengan kurikulum **3-Stage GRPO** (`Stage 1 -> Stage 2 -> Stage 3`).
3. **Studi Ablasi 3 Dimensi pada Tabel 3 & Tabel 4 (`RQ2`)**:
   - `w/o Lin` (`--ablate_dim lin`)
   - `w/o Con` (`--ablate_dim con`)
   - `w/o Emo` (`--ablate_dim emo`)

*Setiap eksperimen yang sudah memiliki `eval_results.json` di Google Drive akan langsung dilewati (auto-resume).*

In [ ]:
assert torch.cuda.is_available(), "Aktifkan GPU (A100 / L4) melalui Runtime > Change runtime type > GPU sebelum menjalankan sel ini."

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import train_idsarcasm_grpo as grpo_mod

all_experiment_results = {}

for ds_key in DATASETS_TO_RUN:
    ds_data_dir = DATA_DIR / ds_key
    ds_run_dir = RUN_DIR / ds_key
    ds_run_dir.mkdir(parents=True, exist_ok=True)
    all_experiment_results[ds_key] = {}

    train_csv = str(ds_data_dir / "train_grpo.csv")
    val_csv = str(ds_data_dir / "validation.csv")
    test_csv = str(ds_data_dir / "test.csv")

    # ------------------------------------------------------------------
    # A. Evaluasi Baseline LLM (0-shot, 5-shot, 0-shot CoT, SFT) - Tabel 2
    # ------------------------------------------------------------------
    if RUN_ZERO_AND_FEW_SHOT_BASELINES:
        for base_variant in ["Qwen 0-shot", "Qwen 5-shot", "Qwen 0-shot CoT", "Qwen SFT"]:
            variant_slug = base_variant.lower().replace(" ", "_").replace("-", "_")
            variant_out = ds_run_dir / variant_slug
            variant_out.mkdir(parents=True, exist_ok=True)
            eval_file = variant_out / "eval_results.json"

            if eval_file.exists():
                print(f"[{ds_key} | {base_variant}] Sudah selesai, memuat dari Drive.")
                all_experiment_results[ds_key][base_variant] = json.loads(eval_file.read_text(encoding="utf-8"))
                continue

            print(f"\n>>> Menjalankan [{ds_key} | {base_variant}]...")
            use_bf16 = torch.cuda.is_bf16_supported()
            tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            model = AutoModelForCausalLM.from_pretrained(
                BASE_MODEL,
                torch_dtype=torch.bfloat16 if use_bf16 else torch.float16,
                device_map="auto",
            )

            train_df = pd.read_csv(train_csv)
            few_shots = None
            direct_zs = False

            if base_variant == "Qwen 0-shot":
                direct_zs = True
            elif base_variant == "Qwen 5-shot":
                few_shots = train_df.head(5).to_dict(orient="records")
            elif base_variant == "Qwen SFT":
                peft_cfg = LoraConfig(r=16, lora_alpha=16, target_modules="all-linear", task_type="CAUSAL_LM")
                model = get_peft_model(model, peft_cfg)
                sft_rows = [{"text": f"{r['question']}\n\n{{ {r['answer']} }}"} for _, r in train_df.iterrows()]
                sft_args = TrainingArguments(
                    output_dir=str(variant_out / "ckpt"),
                    per_device_train_batch_size=4,
                    gradient_accumulation_steps=4,
                    learning_rate=2e-5,
                    max_steps=80,
                    bf16=use_bf16,
                    fp16=not use_bf16,
                    logging_steps=20,
                    save_strategy="no",
                    report_to="none",
                    seed=SEED,
                )
                trainer = SFTTrainer(model=model, train_dataset=Dataset.from_list(sft_rows), args=sft_args)
                trainer.train()
                del trainer

            val_m, _ = grpo_mod.evaluate_model_on_csv(
                model, tokenizer, val_csv, few_shot_examples=few_shots, direct_zero_shot=direct_zs
            )
            test_m, test_logs = grpo_mod.evaluate_model_on_csv(
                model, tokenizer, test_csv, few_shot_examples=few_shots, direct_zero_shot=direct_zs
            )
            res_obj = {
                "model": base_variant,
                "dataset": ds_key,
                "seed": SEED,
                "validation": val_m,
                "test": test_m,
            }
            eval_file.write_text(json.dumps(res_obj, indent=2), encoding="utf-8")
            (variant_out / "sample_predictions.json").write_text(
                json.dumps(test_logs, indent=2, ensure_ascii=False), encoding="utf-8"
            )
            all_experiment_results[ds_key][base_variant] = res_obj
            del model
            gc.collect()
            torch.cuda.empty_cache()

    # ------------------------------------------------------------------
    # B. Pelatihan 3-Stage GRPO Sarcasm-R1 (Full) - Tabel 2 & Tabel 3
    # ------------------------------------------------------------------
    r1_out = ds_run_dir / "sarcasm_r1_full"
    r1_eval_file = r1_out / "eval_results.json"
    if r1_eval_file.exists():
        print(f"[{ds_key} | Sarcasm-R1] Sudah selesai, memuat dari Drive.")
        all_experiment_results[ds_key]["Sarcasm-R1"] = json.loads(r1_eval_file.read_text(encoding="utf-8"))
    else:
        print(f"\n>>> Melatih 3-Stage GRPO [{ds_key} | Sarcasm-R1 (Full)]...")
        r1_res = grpo_mod.run_three_stage_grpo(
            model_name_or_path=BASE_MODEL,
            train_csv=train_csv,
            val_csv=val_csv,
            test_csv=test_csv,
            output_dir=str(r1_out),
            ablate_dim="none",
            reward_schedule="paper_3stage",
            stage1_steps=STAGE1_STEPS,
            stage2_steps=STAGE2_STEPS,
            stage3_steps=STAGE3_STEPS,
            seed=SEED,
        )
        all_experiment_results[ds_key]["Sarcasm-R1"] = r1_res
        gc.collect()
        torch.cuda.empty_cache()

    # ------------------------------------------------------------------
    # C. Studi Ablasi 3 Dimensi (w/o Lin, w/o Con, w/o Emo) - Tabel 3 & 4
    # ------------------------------------------------------------------
    if ds_key == "twitter" and RUN_ABLATION_STUDY_ON_TWITTER:
        for ablate_key, ablate_label in [("lin", "w/o Lin"), ("con", "w/o Con"), ("emo", "w/o Emo")]:
            abl_out = ds_run_dir / f"ablation_wo_{ablate_key}"
            abl_eval_file = abl_out / "eval_results.json"
            if abl_eval_file.exists():
                print(f"[{ds_key} | {ablate_label}] Sudah selesai, memuat dari Drive.")
                all_experiment_results[ds_key][ablate_label] = json.loads(abl_eval_file.read_text(encoding="utf-8"))
            else:
                print(f"\n>>> Menjalankan Ablasi [{ds_key} | {ablate_label}]...")
                abl_res = grpo_mod.run_three_stage_grpo(
                    model_name_or_path=BASE_MODEL,
                    train_csv=train_csv,
                    val_csv=val_csv,
                    test_csv=test_csv,
                    output_dir=str(abl_out),
                    ablate_dim=ablate_key,
                    reward_schedule="paper_3stage",
                    stage1_steps=STAGE1_STEPS // 2,
                    stage2_steps=ABLATION_STEPS // 2,
                    stage3_steps=ABLATION_STEPS,
                    seed=SEED,
                )
                all_experiment_results[ds_key][ablate_label] = abl_res
                gc.collect()
                torch.cuda.empty_cache()

print("\nSeluruh eksperimen selesai dan tersimpan di Google Drive!")

## 9. Kompilasi Otomatis Tabel 2, Tabel 3, Tabel 4, Grafik, & Laporan Tesis (`laporan_sarcasm_r1.md`)

Sel ini menggabungkan hasil eksperimen **Sarcasm-R1** Anda dengan seluruh baseline resmi dari paper **IdSarcasm (IEEE Access 2024)**, hasil **Replikasi IndoBERT & XLM-R**, serta **HAMR**, lalu mengekspor laporan lengkap siap pakai untuk Bab 4 Tesis S2 Anda.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from train_idsarcasm_grpo import compute_table4_significance

# Acuan Baseline Resmi dari Paper IdSarcasm (Tabel 1 & Tabel 2, Hal. 6) & Eksperimen Sebelumnya
PAPER_BASELINES = {
    "twitter": [
        {"Model": "Logistic Regression (Paper)", "Acc": 0.8661, "F1_Sarc": 0.7142, "Prec": 0.7627, "Rec": 0.6716, "Ma_F1": None},
        {"Model": "SVM (Paper)", "Acc": 0.8624, "F1_Sarc": 0.6782, "Prec": 0.8125, "Rec": 0.5820, "Ma_F1": None},
        {"Model": "IndoNLU IndoBERT Base (Paper)", "Acc": 0.8662, "F1_Sarc": 0.7273, "Prec": 0.7385, "Rec": 0.7164, "Ma_F1": None},
        {"Model": "XLM-R Large (Paper)", "Acc": 0.8885, "F1_Sarc": 0.7692, "Prec": 0.7937, "Rec": 0.7463, "Ma_F1": None},
        {"Model": "BLOOMZ-7.1B Zero-Shot (Paper)", "Acc": 0.2531, "F1_Sarc": 0.3968, "Prec": 0.2483, "Rec": 0.9865, "Ma_F1": None},
        {"Model": "mT0-XL Zero-Shot (Paper)", "Acc": 0.2490, "F1_Sarc": 0.3988, "Prec": 0.2490, "Rec": 1.0000, "Ma_F1": None},
        {"Model": "IndoBERT Base (Replikasi Seed 42)", "Acc": 0.8903, "F1_Sarc": 0.7611, "Prec": 0.8319, "Rec": 0.7015, "Ma_F1": None},
        {"Model": "IndoBERT + HAMR (Mean 3 Seed)", "Acc": 0.8488, "F1_Sarc": 0.6713, "Prec": 0.7345, "Rec": 0.6219, "Ma_F1": 0.7865},
    ],
    "reddit": [
        {"Model": "Logistic Regression (Paper)", "Acc": 0.7829, "F1_Sarc": 0.4887, "Prec": 0.5943, "Rec": 0.4150, "Ma_F1": None},
        {"Model": "SVM (Paper)", "Acc": 0.7885, "F1_Sarc": 0.4467, "Prec": 0.6461, "Rec": 0.3413, "Ma_F1": None},
        {"Model": "IndoNLU IndoBERT Base (Paper)", "Acc": 0.7921, "F1_Sarc": 0.6100, "Prec": 0.5745, "Rec": 0.6501, "Ma_F1": None},
        {"Model": "XLM-R Large (Paper)", "Acc": 0.8120, "F1_Sarc": 0.6274, "Prec": 0.6217, "Rec": 0.6331, "Ma_F1": None},
        {"Model": "BLOOMZ-7.1B Zero-Shot (Paper)", "Acc": 0.3058, "F1_Sarc": 0.4036, "Prec": 0.2575, "Rec": 0.9388, "Ma_F1": None},
        {"Model": "mT0-XL Zero-Shot (Paper)", "Acc": 0.2503, "F1_Sarc": 0.4001, "Prec": 0.2500, "Rec": 1.0000, "Ma_F1": None},
        {"Model": "IndoBERT Base (Replikasi Seed 42)", "Acc": 0.7921, "F1_Sarc": 0.6125, "Prec": 0.5735, "Rec": 0.6572, "Ma_F1": None},
        {"Model": "XLM-R Large (Replikasi Seed 42)", "Acc": 0.8102, "F1_Sarc": 0.5946, "Prec": 0.6380, "Rec": 0.5567, "Ma_F1": None},
    ],
}

comparison_tables = {}
for ds_key in DATASETS_TO_RUN:
    rows = list(PAPER_BASELINES.get(ds_key, []))
    for exp_name, exp_data in all_experiment_results.get(ds_key, {}).items():
        if exp_name.startswith("w/o"):
            continue
        tm = exp_data["test"]
        rows.append({
            "Model": exp_name,
            "Acc": tm["accuracy"],
            "F1_Sarc": tm["f1_sarcastic"],
            "Prec": tm["precision_sarcastic"],
            "Rec": tm["recall_sarcastic"],
            "Ma_F1": tm["macro_f1"],
        })
    df_comp = pd.DataFrame(rows)
    comparison_tables[ds_key] = df_comp
    print(f"\n=== Tabel 2 ({ds_key.upper()}): Perbandingan Metode pada Test Set Resmi ===")
    display(df_comp)

# Tabel 3: Hasil Studi Ablasi 3 Dimensi pada Twitter (Seperti Tabel 3 Paper Sarcasm-R1)
if "twitter" in all_experiment_results:
    abl_rows = []
    for m_name in ["w/o Lin", "w/o Con", "w/o Emo", "Sarcasm-R1"]:
        if m_name in all_experiment_results["twitter"]:
            tm = all_experiment_results["twitter"][m_name]["test"]
            abl_rows.append({
                "Method": m_name,
                "Acc (%)": tm["Acc_pct"],
                "Ma-F1 (%)": tm["Ma_F1_pct"],
                "F1 Sarkastik (Binary)": tm["f1_sarcastic"],
                "Precision Sarkastik": tm["precision_sarcastic"],
                "Recall Sarkastik": tm["recall_sarcastic"],
            })
    if abl_rows:
        df_abl = pd.DataFrame(abl_rows)
        print("\n=== Tabel 3 (RQ2): Hasil Studi Ablasi Reward Model 3 Dimensi (Language, Context, Emotion) ===")
        display(df_abl)

# Simpan ringkasan lengkap ke Google Drive
summary_report_path = ROOT_OUT / "ringkasan_eksperimen_sarcasm_r1.json"
summary_report_path.write_text(json.dumps(all_experiment_results, indent=2, ensure_ascii=False), encoding="utf-8")
print("Ringkasan JSON disimpan ke:", summary_report_path)

# Auto-Disconnect GPU A100 untuk menghemat Compute Unit Google Colab Pro
if AUTO_DISCONNECT_A100:
    try:
        from google.colab import runtime
        print("Seluruh output telah aman di Google Drive. Memutus runtime A100 secara otomatis untuk menghemat Compute Unit...")
        runtime.unassign()
    except Exception:
        pass